# Haptic Ground Truth — Colab Pipeline

Same four algorithms as **[Sound2Hap](https://github.com/Iris1215/Sound2Hap)** (CHI 2026), with **frozen multimodal context detection** before haptic synthesis.

Convert **3–5 minute** video into **gated candidate haptic tracks** for human-in-the-loop evaluation.

**Runtime:** **T4 GPU** recommended for context detection (AST + ViViT). Sound2Hap A–D can run on CPU.

**Output:** mono **8 kHz** haptic WAV files + `events.json`

**Setup:** Run all cells top-to-bottom. Upload a video when prompted — no Google Drive needed.

| Phase | Component |
|-------|-----------|
| **1** | Tokenization (100Hz) + Frozen Context Detectors + AST/ViViT encoders |
| **2** | Frozen fusion → `events.json` → gated audio |
| **3** | Sound2Hap A–D |

In [ ]:
# Install system + Python dependencies
!apt-get -qq install -y ffmpeg > /dev/null
!pip install -q numpy scipy librosa soundfile audioread resampy torch torchaudio matplotlib mosqito pyyaml transformers accelerate decord av opencv-python Pillow


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/haptic-groundtruth")
PKG_DIR = PROJECT_ROOT / "haptic_gt"
PKG_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "__init__.py": """\"\"\"Ground-truth haptic track generation from video audio.\"\"\"

from .pipeline import generate_candidate_tracks

__all__ = ["generate_candidate_tracks"]
""",
    "algorithms/__init__.py": """\"\"\"Sound2Hap signal-processing algorithms.\"\"\"
""",
    "algorithms/freq_shift.py": """\"\"\"
Frequency shifting audio-to-vibration (Sound2Hap / Okazaki et al., 2015).

Adapted from: https://github.com/Iris1215/Sound2Hap
\"\"\"

from __future__ import annotations

from pathlib import Path
from typing import Union

import librosa
import numpy as np
import soundfile as sf
import torch
from scipy.signal import butter, lfilter

from haptic_gt.utils.normalization import normalize_audio

VIB_SR = 8000


def _butter_bandpass(sr: int, center_hz: float = 250.0, q: float = 1.0, order: int = 4):
    bw = center_hz / q
    low_hz = max(center_hz - bw / 2, 1.0)
    high_hz = min(center_hz + bw / 2, sr / 2 - 1)
    wn = [low_hz / (sr / 2), high_hz / (sr / 2)]
    return butter(order, wn, btype="band")


def _butter_highpass(sr: int, cutoff_hz: float = 10.0, order: int = 2):
    wn = cutoff_hz / (sr / 2)
    return butter(order, wn, btype="high")


def process_file(
    in_wav: Union[str, Path],
    out_wav: Union[str, Path],
    centre_hz: float = 250.0,
    q: float = 1.0,
) -> None:
    y, sr = librosa.load(in_wav, sr=None, mono=True)

    wav_tensor = torch.from_numpy(y).float().unsqueeze(0)
    y_norm_t = normalize_audio(wav_tensor, normalize=True, strategy="peak")
    y = y_norm_t.squeeze(0).numpy()

    y_1ot = librosa.effects.pitch_shift(y, sr=sr, n_steps=-12, res_type="kaiser_best")
    y_2ot = librosa.effects.pitch_shift(y, sr=sr, n_steps=-24, res_type="kaiser_best")
    mix = y + y_1ot + y_2ot

    rms = np.sqrt(np.mean(mix**2) + 1e-12)
    mix /= rms * np.sqrt(2)

    b_hp, a_hp = _butter_highpass(sr, cutoff_hz=10.0)
    mix = lfilter(b_hp, a_hp, mix)

    b, a = _butter_bandpass(sr, centre_hz, q)
    mix_bp = lfilter(b, a, mix)
    mix_bp = librosa.resample(mix_bp, orig_sr=sr, target_sr=VIB_SR)
    mix_bp = np.clip(mix_bp, -1.0, 1.0)

    out_path = Path(out_wav)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(out_path, mix_bp.astype(np.float32), VIB_SR, subtype="PCM_16")
""",
    "algorithms/haptic_gen.py": """\"\"\"
HapticGen-style RMS-driven NCO synthesis (Sound2Hap / Sung et al., CHI 2025).

Adapted from: https://github.com/Iris1215/Sound2Hap
\"\"\"

from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import soundfile as sf
import torch

from haptic_gt.utils.normalization import normalize_audio

WANTED_BIN_SIZE_SEC = 0.010
BASE_FREQ = 200.0
VIB_SR = 8000


def amp_env_on_wav_norm(
    wav_norm: np.ndarray,
    input_sample_rate: int,
    output_sample_rate: int,
) -> np.ndarray:
    wav_norm = wav_norm.squeeze()
    num_samples = len(wav_norm)
    duration_sec = num_samples / input_sample_rate
    samples_per_bin = int(WANTED_BIN_SIZE_SEC * input_sample_rate)
    num_bins = num_samples // samples_per_bin

    wav_chunks = np.array_split(wav_norm, num_bins)
    rms_bins = np.array([np.sqrt(np.mean(chunk**2)) for chunk in wav_chunks])
    rms_max = np.max(rms_bins)
    rms_norm = np.sqrt(2)
    rms_amplify = max(1.0, min(1.2, 1.0 / (rms_max * rms_norm)))
    rms_norm_amp = rms_norm * rms_amplify
    out_samples = int(duration_sec * output_sample_rate)

    phase_acc = 0.0
    output = np.zeros(out_samples)
    for i in range(out_samples):
        t = i / output_sample_rate
        t_prog = t / duration_sec
        bin_fi = t_prog * num_bins
        bin_lo = int(bin_fi)
        bin_hi = min(num_bins - 1, int(math.ceil(bin_fi)))
        bin_fr = bin_fi - bin_lo
        rms_val = (
            rms_bins[bin_lo] * (1.0 - bin_fr) + rms_bins[bin_hi] * bin_fr
        ) * rms_norm_amp
        freq_offset = (rms_val - 0.3) * 100.0
        phase_delta = 2.0 * math.pi * (BASE_FREQ + freq_offset) / output_sample_rate
        phase_acc = (phase_acc + phase_delta) % (2.0 * math.pi)
        output[i] = rms_val * math.sin(phase_acc)

    return output


def process_file(input_path: str | Path, output_path: str | Path) -> None:
    wav_data, sr = sf.read(input_path)
    wav_tensor = torch.from_numpy(wav_data).float().unsqueeze(0)
    wav_norm_tensor = normalize_audio(
        wav_tensor,
        normalize=True,
        strategy="peak",
        peak_clip_headroom_db=0,
        peak_normalize_db_clamp=0,
    )
    wav = wav_norm_tensor.squeeze(0).numpy()
    env_signal = amp_env_on_wav_norm(wav, sr, VIB_SR)

    out_path = Path(output_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(out_path, env_signal, VIB_SR, subtype="PCM_16")
""",
    "algorithms/percept.py": """\"\"\"
Perception-level audio-to-vibration translator (Sound2Hap / Lee & Choi, CHI 2013).

Adapted from: https://github.com/Iris1215/Sound2Hap
\"\"\"

from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
from scipy.signal import find_peaks

from haptic_gt.utils.normalization import normalize_audio

AUDIO_SR = 44100
VIB_SR = 8000
FRAME_S = 4096
F1 = 175.0
F2 = 210.0
CR = 0.035
OR = 0.40
CV = 1
CL = 0.1
OL = 3.8
C_FULLBAND = 0.065
F_FULLBAND = 6400
C_BASS = 1.91
F_BASS = 200
C = 1.37

_ISO_FREQ = np.array(
    [
        25, 31.5, 40, 50, 63, 80, 100, 125, 160, 200, 250, 315, 400, 500, 630,
        800, 1000, 1250, 1600, 2000, 2500, 3150, 4000, 5000, 6300,
    ]
)
_ISO_SPL60 = np.array(
    [
        104.23, 99.08, 94.18, 89.96, 85.94, 82.05, 78.65, 75.56, 72.47, 69.86,
        67.53, 65.39, 63.45, 62.05, 60.81, 59.89, 60.01, 62.15, 63.19, 59.96,
        57.26, 56.42, 57.57, 60.89, 66.36,
    ]
)


def iso60phon(f: np.ndarray) -> np.ndarray:
    return np.interp(f, _ISO_FREQ, _ISO_SPL60, left=_ISO_SPL60[0], right=_ISO_SPL60[-1])


def auditory_loudness(frame: np.ndarray, content: str) -> float:
    if content == "music":
        c_use, f_max = C_BASS, F_BASS
    else:
        c_use, f_max = C_FULLBAND, F_FULLBAND

    mag = np.abs(np.fft.rfft(frame))
    freqs = np.fft.rfftfreq(frame.size, 1 / AUDIO_SR)
    mask = (freqs >= 25) & (freqs <= f_max)
    mag = mag[mask]
    freqs = freqs[mask]

    db = 20 * np.log10(C * mag + 1e-12)
    af = iso60phon(freqs)
    loudness = c_use * np.sum(db / af)
    return max(0.0, loudness)


def auditory_roughness(frame: np.ndarray, peak_db: float = -40.0) -> float:
    mag = np.abs(np.fft.rfft(frame))
    freqs = np.fft.rfftfreq(frame.size, 1 / AUDIO_SR)
    mask = (freqs >= 25) & (freqs <= 6400)
    mag = mag[mask]
    freqs = freqs[mask]

    db = 20 * np.log10(mag + 1e-12)
    thresh = db.max() + peak_db
    peaks, _ = find_peaks(db, height=thresh)

    f = freqs[peaks]
    x = mag[peaks]
    roughness = 0.0
    for i in range(len(f)):
        for j in range(i + 1, len(f)):
            f1, f2 = f[i], f[j]
            x1, x2 = x[i], x[j]
            xm, xM = min(x1, x2), max(x1, x2)
            fd = abs(f2 - f1)
            s = 0.24 / (0.0207 * min(f1, f2) + 18.96)
            term = ((xm * xM) ** 0.1 / 2.0) * (2 * xm / (xm + xM)) ** 3.11
            roughness += term * (math.exp(-3.5 * s * fd) - math.exp(-5.75 * s * fd))
    return roughness


def perceptual_targets(la: float, ra: float, content: str) -> tuple[float, float]:
    if content == "music":
        iv = CL * la - OL
    else:
        iv = CR * math.sqrt(la) * (ra**2) - OR
    rv = CV * ra
    return max(0, iv), rv


def amplitudes_from_percepts(iv: float, rv: float) -> tuple[float, float]:
    if iv <= 0.0:
        return 0.0, 0.0

    rv_max = (801.0 / 113.0) + 0.529 * iv + 0.479
    rv_adj = min(rv, rv_max)
    disc = max(0.0, 801.0 - 113.0 * (rv_adj - 0.529 * iv - 0.479))
    r1 = (28.3 + math.sqrt(disc)) / 56.3
    r2 = (28.3 - math.sqrt(disc)) / 56.3
    valid = [s for s in (r1, r2) if 0.0 <= s <= 1.0]
    s = min(valid) if valid else (28.3 / 56.3)

    a = ((25.8 * s**2 - 25.5 * s + rv_adj - 0.203) / 3.98) ** 2
    a2 = a * s
    a1 = a - a2
    return a1, a2


def synth_vibration(a1: float, a2: float, n_samples: int) -> np.ndarray:
    t = np.arange(n_samples) / VIB_SR
    return a1 * np.sin(2 * math.pi * F1 * t) + a2 * np.sin(2 * math.pi * F2 * t)


def read_wav_mono_44k(fname: str | Path) -> np.ndarray:
    wav_data, _ = sf.read(fname)
    if wav_data.ndim > 1:
        wav_data = wav_data.mean(axis=1)
    wav_tensor = torch.from_numpy(wav_data).float().unsqueeze(0)
    wav_norm_tensor = normalize_audio(
        wav_tensor,
        normalize=True,
        strategy="peak",
        peak_clip_headroom_db=0,
        peak_normalize_db_clamp=0,
    )
    return wav_norm_tensor.squeeze(0).numpy().astype("float32")


def process_file(
    in_wav: str | Path,
    out_wav: str | Path,
    content: str = "game",
) -> None:
    audio = read_wav_mono_44k(in_wav)
    hop_s = FRAME_S
    n_out_total = int(np.ceil(len(audio) * VIB_SR / AUDIO_SR))
    vib_full = np.zeros(n_out_total, dtype=np.float32)

    for start in range(0, len(audio), hop_s):
        block = audio[start : start + FRAME_S]
        if block.size == 0:
            break
        if block.size < FRAME_S:
            block = np.pad(block, (0, FRAME_S - block.size), "constant")

        la = auditory_loudness(block, content)
        ra = auditory_roughness(block)
        iv, rv = perceptual_targets(la, ra, content)
        a1, a2 = amplitudes_from_percepts(iv, rv)

        n_out = int(round(FRAME_S * VIB_SR / AUDIO_SR))
        vib_seg = synth_vibration(a1, a2, n_out)
        rms_seg = np.sqrt(np.mean(vib_seg**2) + 1e-12)
        vib_seg /= rms_seg * np.sqrt(2)

        out_start = int(round(start * VIB_SR / AUDIO_SR))
        out_end = out_start + n_out
        if out_end > n_out_total:
            vib_full[out_start:] += vib_seg[: n_out_total - out_start]
        else:
            vib_full[out_start:out_end] += vib_seg

    vib_full = np.clip(vib_full, -1.0, 1.0)
    out_path = Path(out_wav)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(out_path, vib_full, VIB_SR, subtype="PCM_16")
""",
    "algorithms/pitch_match.py": """\"\"\"
Pitch Match audio-to-vibration (Sound2Hap / Kim et al., IEEE ToH 2023).

Python version used for Sound2Hap web tool. MATLAB version used in the study.

Adapted from: https://github.com/Iris1215/Sound2Hap
\"\"\"

from __future__ import annotations

from dataclasses import dataclass
from fractions import Fraction
from pathlib import Path

import numpy as np
import soundfile as sf
from scipy.signal import get_window, resample_poly

try:
    from mosqito.functions.loudness_zwtv._loudness_zwtv import loudness_zwtv

    MOSQITO_AVAILABLE = True
except Exception:
    MOSQITO_AVAILABLE = False

VIB_SR = 8000


@dataclass
class Config:
    regressionCoeffs: dict
    vibrationFreqRange: tuple
    binSizeMs: float
    overlapRatio: float
    smoothingWindow: int
    inputSampleRate: int
    outputSampleRate: int


def get_config() -> Config:
    return Config(
        regressionCoeffs={2: -0.005, 3: 0.003, 9: -0.015, 12: 0.008, 24: 0.008},
        vibrationFreqRange=(50.0, 398.0),
        binSizeMs=10.0,
        overlapRatio=0.5,
        smoothingWindow=3,
        inputSampleRate=44100,
        outputSampleRate=VIB_SR,
    )


def normalize_audio(audio: np.ndarray, do_normalize: bool) -> np.ndarray:
    scale_peak = 10 ** (-1 / 20)
    normalize_peak = 1.0
    wav_max = np.max(np.abs(audio)) + 1e-12
    rescaling = min(max(1.0, normalize_peak / wav_max), scale_peak / wav_max)
    if do_normalize or (rescaling < 1.0):
        audio = audio * rescaling
    return audio


def rms(x: np.ndarray) -> float:
    return float(np.sqrt(np.mean(np.square(x)) + 1e-12))


def _specific_and_total_loudness_bark(audio_bin: np.ndarray, sr: int):
    if not MOSQITO_AVAILABLE:
        env = np.abs(audio_bin)
        total = float(np.mean(env))
        spec24 = np.zeros(24, dtype=np.float32)
        spec24[0] = total
        return spec24, total

    try:
        results = loudness_zwtv(audio_bin, sr, field_type="free")
        n_time = np.asarray(results["N"]).reshape(-1)
        n_spec = np.asarray(results["N_specific"])
        total_loudness = float(np.mean(n_time)) if n_time.size else 0.0

        if n_spec.ndim == 2 and n_spec.shape[1] >= 240:
            spec_time_mean = np.mean(n_spec, axis=0)
            spec24 = np.zeros(24, dtype=np.float32)
            for i in range(24):
                start = i * 10
                end = start + 10
                spec24[i] = float(np.sum(spec_time_mean[start:end]))
        else:
            if n_spec.ndim == 1:
                vec = n_spec
            else:
                vec = np.mean(n_spec, axis=0) if n_spec.size else np.zeros(240)
            idx = np.linspace(0, len(vec) - 1, 24)
            spec24 = np.interp(idx, np.arange(len(vec)), vec).astype(np.float32)

        spec24[~np.isfinite(spec24)] = 0.0
        return spec24, total_loudness
    except Exception:
        env = np.abs(audio_bin)
        total = float(np.mean(env))
        spec24 = np.zeros(24, dtype=np.float32)
        spec24[0] = total
        return spec24, total


def predict_vibration_frequency(specific_loudness_24: np.ndarray, cfg: Config) -> float:
    predicted = 0.0
    for bark_band, coeff in cfg.regressionCoeffs.items():
        idx = int(bark_band) - 1
        if 0 <= idx < len(specific_loudness_24):
            predicted += coeff * float(specific_loudness_24[idx]) * 1000.0
    predicted = abs(predicted)
    vmin, vmax = cfg.vibrationFreqRange
    return float(np.clip(predicted, vmin, vmax))


def analyze_audio_bins(audio: np.ndarray, sr: int, cfg: Config):
    bin_size = int(round(cfg.binSizeMs * sr / 1000.0))
    hop = max(1, int(round(bin_size * (1.0 - cfg.overlapRatio))))
    if bin_size < 2:
        bin_size = 2
    starts = np.arange(0, max(1, len(audio) - bin_size + 1), hop, dtype=int)
    if starts.size == 0:
        starts = np.array([0], dtype=int)

    times = (starts + bin_size / 2.0) / float(sr)
    freqs = np.zeros(starts.size, dtype=np.float32)
    amps = np.zeros(starts.size, dtype=np.float32)
    win = get_window("hann", bin_size, fftbins=False).astype(np.float32)

    for i, s in enumerate(starts):
        e = min(s + bin_size, len(audio))
        chunk = np.zeros(bin_size, dtype=np.float32)
        seg = audio[s:e]
        chunk[: len(seg)] = seg
        chunk *= win

        if rms(chunk) < 1e-3:
            freqs[i] = freqs[i - 1] if i > 0 else np.mean(cfg.vibrationFreqRange)
            amps[i] = 0.0
            continue

        spec24, loud = _specific_and_total_loudness_bark(chunk, sr)
        freqs[i] = predict_vibration_frequency(spec24, cfg)
        amps[i] = float(loud)

    if cfg.smoothingWindow > 1 and len(freqs) > cfg.smoothingWindow:
        k = cfg.smoothingWindow
        kernel = np.ones(k, dtype=np.float32) / k
        freqs = np.convolve(freqs, kernel, mode="same")

    return times.astype(np.float64), freqs.astype(np.float64), amps.astype(np.float64)


def generate_time_varying_vibration(audio: np.ndarray, sr: int, cfg: Config):
    bin_t, bin_f, bin_a = analyze_audio_bins(audio, sr, cfg)
    t = np.arange(len(audio), dtype=np.float64) / float(sr)

    if len(bin_t) == 1:
        f_inst = np.full_like(t, bin_f[0], dtype=np.float64)
        a_inst = np.full_like(t, bin_a[0], dtype=np.float64)
    else:
        try:
            from scipy.interpolate import PchipInterpolator

            f_inst = PchipInterpolator(bin_t, bin_f, extrapolate=True)(t)
        except Exception:
            f_inst = np.interp(t, bin_t, bin_f, left=bin_f[0], right=bin_f[-1])
        a_inst = np.interp(t, bin_t, bin_a, left=bin_a[0], right=bin_a[-1])

    rms_current = rms(a_inst)
    rms_norm = np.sqrt(2.0)
    rms_amplify = max(1.0, min(1.2, 1.0 / (rms_current * rms_norm))) if rms_current > 0 else 1.0
    a_inst = a_inst * (rms_norm * rms_amplify) if rms_current > 0 else np.full_like(t, 0.1)

    dt = 1.0 / float(sr)
    phi = np.empty_like(t)
    phi[0] = 0.0
    phi[1:] = 2.0 * np.pi * np.cumsum(f_inst[:-1]) * dt
    v = a_inst * np.sin(phi)

    fade_len = int(round(0.01 * sr))
    if len(v) > 2 * fade_len and fade_len > 0:
        fade_in = np.linspace(0.0, 1.0, fade_len)
        fade_out = np.linspace(1.0, 0.0, fade_len)
        v[:fade_len] *= fade_in
        v[-fade_len:] *= fade_out

    return v.astype(np.float32), f_inst.astype(np.float32), a_inst.astype(np.float32)


def generate_vibration_signal(audio: np.ndarray, sr: int, cfg: Config):
    v, f_arr, a_arr = generate_time_varying_vibration(audio, sr, cfg)
    analysis_info = {
        "method": "time_varying",
        "duration": len(audio) / float(sr),
        "freqMean": float(np.mean(f_arr)),
        "freqRange": (float(np.min(f_arr)), float(np.max(f_arr))),
        "freqStd": float(np.std(f_arr)),
    }
    return v, f_arr, a_arr, analysis_info


def _read_mono(path: str | Path):
    x, sr = sf.read(path, always_2d=False)
    x = x.astype(np.float32)
    if x.ndim == 2:
        x = x.mean(axis=1)
    return x, sr


def _write_int16_wav(path: str | Path, y: np.ndarray, sr: int):
    y = y / (np.max(np.abs(y)) + 1e-12)
    sf.write(path, y, sr, subtype="PCM_16")


def process_file(
    input_file: str | Path,
    output_file: str | Path,
    cfg: Config | None = None,
) -> dict:
    cfg = cfg or get_config()
    audio, sr = _read_mono(input_file)
    duration = len(audio) / float(sr)
    audio = normalize_audio(audio, True)

    v, f_arr, a_arr, info = generate_vibration_signal(audio, sr, cfg)
    fs_out = cfg.outputSampleRate
    if sr != fs_out:
        frac = Fraction(fs_out, sr).limit_denominator(1000)
        v = resample_poly(v, frac.numerator, frac.denominator)

    out_path = Path(output_file)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    _write_int16_wav(out_path, v, fs_out)

    return {
        "inputFile": str(input_file),
        "outputFile": str(output_file),
        "duration": duration,
        "originalSr": sr,
        "targetSr": fs_out,
        "analysisInfo": info,
        "mosqitoAvailable": MOSQITO_AVAILABLE,
    }
""",
    "audio_io.py": """\"\"\"Audio extraction, loading, and export (Sound2Hap-compatible rates).\"\"\"

from __future__ import annotations

import shutil
import subprocess
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf

INPUT_SR = 44_100
VIB_SR = 8_000


def _require_ffmpeg() -> str:
    ffmpeg = shutil.which("ffmpeg")
    if ffmpeg is None:
        raise RuntimeError(
            "ffmpeg not found. Install it first "
            "(Colab: !apt-get -qq install ffmpeg)."
        )
    return ffmpeg


def extract_audio_from_video(
    video_path: str | Path,
    output_path: str | Path,
    sr: int = INPUT_SR,
) -> Path:
    \"\"\"Extract mono 16-bit PCM WAV at 44.1 kHz from a video file.\"\"\"
    video_path = Path(video_path)
    output_path = Path(output_path)
    if not video_path.exists():
        raise FileNotFoundError(f"Video not found: {video_path}")

    ffmpeg = _require_ffmpeg()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        ffmpeg,
        "-y",
        "-i",
        str(video_path),
        "-vn",
        "-ac",
        "1",
        "-ar",
        str(sr),
        "-sample_fmt",
        "s16",
        str(output_path),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg failed:\\n{result.stderr}")
    return output_path


def prepare_source_wav(
    audio_path: str | Path,
    output_path: str | Path,
    sr: int = INPUT_SR,
) -> Path:
    \"\"\"Convert/load audio to mono 16-bit PCM WAV at 44.1 kHz.\"\"\"
    audio_path = Path(audio_path)
    output_path = Path(output_path)
    if not audio_path.exists():
        raise FileNotFoundError(f"Audio not found: {audio_path}")

    audio, _ = librosa.load(audio_path, sr=sr, mono=True)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(output_path, np.clip(audio, -1.0, 1.0), sr, subtype="PCM_16")
    return output_path


def save_haptic(path: str | Path, audio: np.ndarray, sr: int = VIB_SR) -> Path:
    \"\"\"Write a mono haptic track as 16-bit PCM WAV (default 8 kHz).\"\"\"
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(path, np.clip(audio, -1.0, 1.0), sr, subtype="PCM_16")
    return path
""",
    "context/__init__.py": """\"\"\"Frozen multimodal context detection for haptic gating.\"\"\"

from haptic_gt.context.detector import EventResult, detect_events
from haptic_gt.context.frozen_fusion import DetectedEvent

__all__ = ["DetectedEvent", "EventResult", "detect_events"]
""",
    "context/context_detectors.py": """\"\"\"Frozen transformer-based context detectors for sudden symbolic events.\"\"\"

from __future__ import annotations

from dataclasses import dataclass

from haptic_gt.context.encoders import EncoderScore
from haptic_gt.context.taxonomy import Taxonomy, load_taxonomy, match_label_to_category


@dataclass
class SymbolicToken:
    time_sec: float
    label: str
    confidence: float
    modality: str  # "audio" | "video"
    category: str | None = None


IMPULSIVE_AUDIO_KEYWORDS = (
    "thunder",
    "explosion",
    "gunshot",
    "gunfire",
    "machine gun",
    "fireworks",
    "bang",
    "boom",
    "artillery",
    "fusillade",
    "cap gun",
)

IMPULSIVE_VIDEO_KEYWORDS = (
    "shooting",
    "explod",
    "fire",
    "smash",
    "hit",
    "crash",
)

SUSTAINED_AUDIO_KEYWORDS = (
    "vehicle",
    "engine",
    "truck",
    "motor vehicle",
    "idling",
    "tank",
    "rain",
    "wind",
    "storm",
)

SUSTAINED_VIDEO_KEYWORDS = (
    "driving",
    "motorcycl",
    "riding",
)


def _is_impulsive_label(label: str, keywords: tuple[str, ...]) -> bool:
    low = label.lower()
    return any(k in low for k in keywords)


def symbolic_tokens_from_scores(
    encoder_scores: list[EncoderScore],
    taxonomy: Taxonomy | None = None,
    *,
    threshold: float | None = None,
) -> list[SymbolicToken]:
    \"\"\"
    Emit symbolic tokens from a single encoder pass.

    Impulsive labels use context_detector_threshold; sustained labels (vehicle,
    weather-non-thunder) use the lower sustained_encoder_threshold so they are
    not silently filtered out.
    \"\"\"
    taxonomy = taxonomy or load_taxonomy()
    impulsive_thresh = threshold if threshold is not None else taxonomy.context_detector_threshold
    sustained_thresh = taxonomy.sustained_encoder_threshold
    tokens: list[SymbolicToken] = []

    for enc in encoder_scores:
        if enc.source == "audio":
            is_impulsive = _is_impulsive_label(enc.label, IMPULSIVE_AUDIO_KEYWORDS)
            is_sustained = _is_impulsive_label(enc.label, SUSTAINED_AUDIO_KEYWORDS)
            modality = "audio"
        else:
            is_impulsive = _is_impulsive_label(enc.label, IMPULSIVE_VIDEO_KEYWORDS)
            is_sustained = _is_impulsive_label(enc.label, SUSTAINED_VIDEO_KEYWORDS)
            modality = "video"

        if not is_impulsive and not is_sustained:
            continue

        effective_thresh = impulsive_thresh if is_impulsive else sustained_thresh
        if enc.score < effective_thresh:
            continue

        cat = match_label_to_category(taxonomy, enc.label, modality)
        if cat is None:
            continue

        tokens.append(
            SymbolicToken(
                time_sec=enc.time_sec,
                label=enc.label,
                confidence=enc.score,
                modality=modality,
                category=cat,
            )
        )

    return tokens


def detect_symbolic_tokens(
    video_path,
    audio_16k,
    video_frames,
    taxonomy: Taxonomy | None = None,
    *,
    window_sec: float = 0.5,
    hop_sec: float = 0.5,
    duration_sec: float | None = None,
    threshold: float | None = None,
    encoder_scores: list[EncoderScore] | None = None,
) -> list[SymbolicToken]:
    \"\"\"
    Emit symbolic tokens for sudden AV events.

    When encoder_scores is provided, filters that list (no extra model calls).
    Otherwise falls back to legacy full-timeline classification (deprecated).
    \"\"\"
    if encoder_scores is not None:
        return symbolic_tokens_from_scores(
            encoder_scores,
            taxonomy=taxonomy,
            threshold=threshold,
        )

    if duration_sec is None:
        raise ValueError("duration_sec is required when encoder_scores is not provided")

    from pathlib import Path

    from haptic_gt.context.encoders import run_encoder_pass

    taxonomy = taxonomy or load_taxonomy()
    scores = run_encoder_pass(
        Path(video_path),
        audio_16k,
        video_frames,
        window_sec=window_sec,
        hop_sec=hop_sec,
        duration_sec=duration_sec,
        full_scan=True,
    )
    return symbolic_tokens_from_scores(scores, taxonomy=taxonomy, threshold=threshold)
""",
    "context/debug_events.py": """\"\"\"Debug helpers: event timing table + per-event audio/metadata extraction.\"\"\"

from __future__ import annotations

import json
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import soundfile as sf

from haptic_gt.audio_io import INPUT_SR


@dataclass
class EventDebugRow:
    event_id: str
    category: str
    label: str
    start_sec: float
    peak_sec: float
    end_sec: float
    duration_sec: float
    confidence: float
    audio_score: float | None
    video_score: float | None
    sources: list[str]
    included_in_gate: bool
    clip_wav: str | None = None
    # Filled by user during diagnosis (optional)
    actual_peak_sec: float | None = None
    timing_note: str | None = None


def load_events_payload(events_json: str | Path) -> dict:
    path = Path(events_json)
    return json.loads(path.read_text(encoding="utf-8"))


def events_to_debug_rows(payload: dict) -> list[EventDebugRow]:
    rows: list[EventDebugRow] = []
    for i, ev in enumerate(payload.get("events", []), start=1):
        event_id = str(ev.get("event_id") or f"event_{i:03d}")
        start = float(ev.get("start_sec", 0.0))
        peak = float(ev.get("peak_sec", start))
        end = float(ev.get("end_sec", peak))
        rows.append(
            EventDebugRow(
                event_id=event_id,
                category=str(ev.get("category", "")),
                label=str(ev.get("label", "")),
                start_sec=start,
                peak_sec=peak,
                end_sec=end,
                duration_sec=max(0.0, end - start),
                confidence=float(ev.get("confidence", 0.0)),
                audio_score=ev.get("audio_score"),
                video_score=ev.get("video_score"),
                sources=list(ev.get("sources") or []),
                included_in_gate=bool(ev.get("included_in_gate", False)),
            )
        )
    return rows


def format_events_table(rows: list[EventDebugRow]) -> str:
    \"\"\"Plain-text timing table for Colab / terminal diagnosis.\"\"\"
    if not rows:
        return "(no events)"

    header = (
        f"{'id':<12} {'cat':<14} {'start':>8} {'peak':>8} {'end':>8} "
        f"{'dur':>6} {'conf':>6} {'audio':>6} {'vid':>6} {'gate':>5} label"
    )
    lines = [header, "-" * len(header)]
    for r in rows:
        audio = f"{r.audio_score:.3f}" if r.audio_score is not None else "-"
        video = f"{r.video_score:.3f}" if r.video_score is not None else "-"
        lines.append(
            f"{r.event_id:<12} {r.category:<14} "
            f"{r.start_sec:8.3f} {r.peak_sec:8.3f} {r.end_sec:8.3f} "
            f"{r.duration_sec:6.3f} {r.confidence:6.3f} {audio:>6} {video:>6} "
            f"{'Y' if r.included_in_gate else 'N':>5} {r.label}"
        )
    return "\\n".join(lines)


def extract_event_debug_clips(
    source_wav: str | Path,
    events_json: str | Path,
    output_dir: str | Path,
    *,
    sample_rate: int = INPUT_SR,
    pad_sec: float = 0.25,
) -> list[EventDebugRow]:
    \"\"\"
    Write one WAV + JSON sidecar per event for accurate listening/diagnosis.

    Layout:
      debug_events/
        event_001_vehicle/
          clip.wav
          meta.json
        ...
        timing_table.txt
        events_debug.json
    \"\"\"
    source_wav = Path(source_wav)
    events_json = Path(events_json)
    output_dir = Path(output_dir)
    debug_root = output_dir / "debug_events"
    debug_root.mkdir(parents=True, exist_ok=True)

    payload = load_events_payload(events_json)
    rows = events_to_debug_rows(payload)

    audio, sr = sf.read(source_wav, always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    audio = audio.astype(np.float32)
    if sr != sample_rate:
        import librosa

        audio = librosa.resample(audio, orig_sr=sr, target_sr=sample_rate)
        sr = sample_rate
    duration_sec = len(audio) / sr

    exported: list[EventDebugRow] = []
    for row in rows:
        folder = debug_root / f"{row.event_id}_{row.category}"
        folder.mkdir(parents=True, exist_ok=True)
        clip_path = folder / "clip.wav"

        s0 = max(0.0, row.start_sec - pad_sec)
        s1 = min(duration_sec, row.end_sec + pad_sec)
        i0 = max(0, int(s0 * sr))
        i1 = min(len(audio), int(s1 * sr))
        clip = audio[i0:i1]
        sf.write(clip_path, clip, sr, subtype="PCM_16")

        meta = asdict(row)
        meta.update(
            {
                "clip_wav": str(clip_path.relative_to(output_dir)),
                "clip_start_sec": round(s0, 3),
                "clip_end_sec": round(s1, 3),
                "pad_sec": pad_sec,
                "sample_rate": sr,
                # User fills these after listening / watching video:
                "actual_peak_sec": None,
                "timing_error_sec": None,
                "note": None,
            }
        )
        (folder / "meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

        row.clip_wav = meta["clip_wav"]
        exported.append(row)

    table = format_events_table(exported)
    (debug_root / "timing_table.txt").write_text(table + "\\n", encoding="utf-8")

    summary = {
        "source_audio": str(source_wav.name),
        "gate_categories_used": payload.get("gate_categories_used", []),
        "instructions": (
            "Compare peak_sec to the real event in the video/audio. "
            "Fill actual_peak_sec and timing_error_sec (= detected_peak - actual_peak) "
            "in each meta.json or reply with a table."
        ),
        "events": [asdict(r) for r in exported],
    }
    (debug_root / "events_debug.json").write_text(
        json.dumps(summary, indent=2),
        encoding="utf-8",
    )
    return exported


def print_event_timing_report(
    events_json: str | Path,
    *,
    source_wav: str | Path | None = None,
    output_dir: str | Path | None = None,
    extract_clips: bool = True,
) -> list[EventDebugRow]:
    \"\"\"Print timing table and optionally extract per-event debug clips.\"\"\"
    payload = load_events_payload(events_json)
    rows = events_to_debug_rows(payload)
    print("=== Detected event timing table ===")
    print(format_events_table(rows))
    print()
    print("Fill actual peaks like:")
    print("  event_001 actual_peak_sec=...  (or 'ok' if correct)")
    print("  event_004 actual_peak_sec=15.20")
    print()

    if extract_clips and source_wav is not None and output_dir is not None:
        exported = extract_event_debug_clips(source_wav, events_json, output_dir)
        print(f"Wrote debug clips under: {Path(output_dir) / 'debug_events'}")
        return exported
    return rows
""",
    "context/detector.py": """\"\"\"Orchestrate Phase 1 + Phase 2 context detection.\"\"\"

from __future__ import annotations

import json
from dataclasses import dataclass, field
from pathlib import Path

from haptic_gt.context.context_detectors import symbolic_tokens_from_scores
from haptic_gt.context.encoders import run_encoder_pass
from haptic_gt.context.frozen_fusion import DetectedEvent, fuse_events
from haptic_gt.context.mask import (
    apply_gate,
    event_included_in_gate,
    events_for_haptic_gate,
    resolve_gate_categories,
)
from haptic_gt.context.onset_refine import refine_event_timing
from haptic_gt.context.proposals import propose_all_windows
from haptic_gt.context.taxonomy import load_taxonomy
from haptic_gt.context.tokenization import tokenize_video_audio

EVENTS_JSON_NAME = "events.json"
GATED_AUDIO_NAME = "gated_audio.wav"


@dataclass
class EventResult:
    events: list[DetectedEvent] = field(default_factory=list)
    no_events_detected: bool = True
    no_haptic_events: bool = True
    timeline_hz: int = 100
    gate_categories_used: list[str] = field(default_factory=list)
    gated_wav: Path | None = None
    events_json: Path | None = None
    haptic_outputs: dict[str, str] = field(default_factory=dict)

    def to_dict(self, *, output_dir: Path | None = None) -> dict:
        taxonomy = load_taxonomy()
        gate_cats = self.gate_categories_used or resolve_gate_categories(taxonomy)

        def _rel(path: str | None) -> str | None:
            if path is None or output_dir is None:
                return path
            try:
                return str(Path(path).relative_to(output_dir))
            except ValueError:
                return path

        event_rows = []
        for i, e in enumerate(self.events, start=1):
            event_rows.append(
                {
                    "event_id": f"event_{i:03d}",
                    "category": e.category,
                    "label": e.label,
                    "start_sec": round(e.start_sec, 3),
                    "peak_sec": round(e.peak_sec, 3),
                    "end_sec": round(e.end_sec, 3),
                    "confidence": round(e.confidence, 4),
                    "context_token": e.context_token,
                    "audio_score": e.audio_score,
                    "video_score": e.video_score,
                    "sources": e.sources,
                    "included_in_gate": event_included_in_gate(
                        e, taxonomy, gate_categories=gate_cats
                    ),
                }
            )

        haptic_out = {
            key: _rel(val) for key, val in self.haptic_outputs.items() if val
        }

        return {
            "no_events_detected": self.no_events_detected,
            "no_haptic_events": self.no_haptic_events,
            "gate_categories_used": gate_cats,
            "timeline_hz": self.timeline_hz,
            "haptic_outputs": haptic_out,
            "events": event_rows,
        }


def detect_events(
    video_path: str | Path,
    source_wav: str | Path,
    output_dir: str | Path | None = None,
    *,
    taxonomy_path: str | Path | None = None,
    window_sec: float | None = None,
    hop_sec: float = 0.25,
    write_gated: bool = True,
    gate_categories: list[str] | None = None,
    full_scan: bool = False,
) -> EventResult:
    \"\"\"
    Run frozen context detection: propose onsets → classify windows → fuse.

    Phase 1: tokenization → onset proposals → targeted AST/ViViT
    Phase 2: frozen fusion → events.json → optional gated_audio.wav
    \"\"\"
    video_path = Path(video_path)
    source_wav = Path(source_wav)
    taxonomy = load_taxonomy(taxonomy_path)
    window_sec = window_sec if window_sec is not None else taxonomy.proposal_window_sec
    gate_cats = resolve_gate_categories(taxonomy, gate_categories)

    tokens_data = tokenize_video_audio(video_path, source_wav, timeline_hz=taxonomy.timeline_hz)

    if full_scan:
        proposal_windows = None
        encoder_scores = run_encoder_pass(
            video_path,
            tokens_data.audio_16k,
            tokens_data.video_frames,
            window_sec=max(0.75, window_sec),
            hop_sec=hop_sec,
            duration_sec=tokens_data.duration_sec,
            full_scan=True,
        )
    else:
        proposal_windows = propose_all_windows(source_wav, taxonomy)
        encoder_scores = run_encoder_pass(
            video_path,
            tokens_data.audio_16k,
            tokens_data.video_frames,
            window_sec=window_sec,
            hop_sec=hop_sec,
            duration_sec=tokens_data.duration_sec,
            proposal_windows=proposal_windows,
            timeline_hz=taxonomy.timeline_hz,
        )

    tokens = symbolic_tokens_from_scores(encoder_scores, taxonomy=taxonomy)

    events = fuse_events(tokens, encoder_scores, taxonomy=taxonomy)
    events = refine_event_timing(events, source_wav, taxonomy)

    gate_events = events_for_haptic_gate(events, taxonomy, gate_categories=gate_cats)
    result = EventResult(
        events=events,
        no_events_detected=len(events) == 0,
        no_haptic_events=len(gate_events) == 0,
        timeline_hz=taxonomy.timeline_hz,
        gate_categories_used=gate_cats,
    )

    if output_dir is not None:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        events_json = output_dir / EVENTS_JSON_NAME
        events_json.write_text(
            json.dumps(result.to_dict(output_dir=output_dir), indent=2),
            encoding="utf-8",
        )
        result.events_json = events_json

        if write_gated and gate_events:
            gated = output_dir / GATED_AUDIO_NAME
            apply_gate(
                source_wav,
                gated,
                events,
                taxonomy=taxonomy,
                gate_categories=gate_cats,
            )
            result.gated_wav = gated
            result.haptic_outputs["gated_audio"] = str(gated)

    return result
""",
    "context/encoders.py": """\"\"\"Frozen AST and ViViT encoders (requires_grad=False).\"\"\"

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np

from haptic_gt.context.proposals import ProposalWindow

AST_MODEL_ID = "MIT/ast-finetuned-audioset-10-10-0.4593"
VIVIT_MODEL_ID = "google/vivit-b-16x2-kinetics400"

_audio_pipe: Any | None = None
_video_pipe: Any | None = None


def _get_device() -> int:
    try:
        import torch

        return 0 if torch.cuda.is_available() else -1
    except Exception:
        return -1


def get_audio_pipeline():
    global _audio_pipe
    if _audio_pipe is None:
        from transformers import pipeline

        _audio_pipe = pipeline(
            "audio-classification",
            model=AST_MODEL_ID,
            device=_get_device(),
        )
    return _audio_pipe


def get_video_pipeline():
    global _video_pipe
    if _video_pipe is None:
        from transformers import pipeline

        _video_pipe = pipeline(
            "video-classification",
            model=VIVIT_MODEL_ID,
            device=_get_device(),
        )
    return _video_pipe


@dataclass
class EncoderScore:
    time_sec: float
    label: str
    score: float
    source: str  # "audio" | "video"


def _top_predictions(raw: list[dict], top_k: int = 5) -> list[tuple[str, float]]:
    out: list[tuple[str, float]] = []
    for item in raw[:top_k]:
        label = str(item.get("label", ""))
        if label.startswith("LABEL_"):
            label = label.replace("LABEL_", "")
        out.append((label, float(item.get("score", 0.0))))
    return out


def classify_audio_window(
    audio_16k: np.ndarray,
    start_sec: float,
    end_sec: float,
    *,
    top_k: int = 5,
) -> list[EncoderScore]:
    \"\"\"Run frozen AST on an audio slice.\"\"\"
    sr = 16_000
    s0 = max(0, int(start_sec * sr))
    s1 = min(len(audio_16k), int(end_sec * sr))
    if s1 - s0 < sr // 10:
        return []

    clip = audio_16k[s0:s1]
    pipe = get_audio_pipeline()
    preds = pipe(clip, top_k=top_k)
    mid = 0.5 * (start_sec + end_sec)
    return [
        EncoderScore(time_sec=mid, label=label, score=score, source="audio")
        for label, score in _top_predictions(preds, top_k)
    ]


def classify_video_clip_file(
    video_path: Path,
    center_sec: float,
    *,
    top_k: int = 5,
) -> list[EncoderScore]:
    \"\"\"
    Run frozen ViViT on a short clip from the video file.

    The HF video-classification pipeline samples frames internally from the
    full file; for temporal windows we extract a sub-clip with ffmpeg when
  possible, otherwise classify the whole file (coarse fallback).
    \"\"\"
    pipe = get_video_pipeline()
    try:
        preds = pipe(str(video_path), top_k=top_k)
    except Exception:
        return []
    return [
        EncoderScore(time_sec=center_sec, label=label, score=score, source="video")
        for label, score in _top_predictions(preds, top_k)
    ]


def classify_video_frames(
    frames: np.ndarray,
    center_sec: float,
    *,
    top_k: int = 5,
) -> list[EncoderScore]:
    \"\"\"Run ViViT on a pre-extracted frame tensor [T, C, H, W] in [0,1].\"\"\"
    if frames.size == 0:
        return []
    try:
        import torch
    except ImportError:
        return []

    pipe = get_video_pipeline()
    # Pipeline expects list of PIL or tensor; pass uint8 frame list
    tensor = torch.from_numpy(frames).float()
    if tensor.ndim != 4:
        return []
    # Sample up to 32 frames uniformly
    n = tensor.shape[0]
    if n > 32:
        idx = np.linspace(0, n - 1, 32).astype(int)
        tensor = tensor[idx]
    # Convert to uint8 HWC list for pipeline compatibility
    frames_u8 = (tensor.permute(0, 2, 3, 1).numpy() * 255.0).clip(0, 255).astype(np.uint8)
    try:
        from PIL import Image

        pil_frames = [Image.fromarray(f) for f in frames_u8]
        preds = pipe(pil_frames, top_k=top_k)
    except Exception:
        return []

    return [
        EncoderScore(time_sec=center_sec, label=label, score=score, source="video")
        for label, score in _top_predictions(preds, top_k)
    ]


def run_encoder_on_windows(
    windows: list[ProposalWindow],
    video_path: Path,
    audio_16k: np.ndarray,
    video_frames: np.ndarray,
    *,
    timeline_hz: int = 100,
    window_sec: float = 1.0,
    top_k: int = 8,
) -> list[EncoderScore]:
    \"\"\"Classify frozen AST + ViViT only at proposal-centered windows.\"\"\"
    if not windows:
        return []

    duration_sec = len(audio_16k) / 16_000
    half = window_sec / 2.0
    scores: list[EncoderScore] = []

    for win in windows:
        t0 = max(0.0, win.center_sec - half)
        t1 = min(duration_sec, win.center_sec + half)
        scores.extend(classify_audio_window(audio_16k, t0, t1, top_k=top_k))

        bin_start = int(t0 * timeline_hz)
        bin_end = int(t1 * timeline_hz)
        bin_end = min(bin_end, len(video_frames))
        if bin_end > bin_start:
            clip = video_frames[bin_start:bin_end]
            scores.extend(
                classify_video_frames(clip, center_sec=win.center_sec, top_k=top_k)
            )
        else:
            scores.extend(
                classify_video_clip_file(video_path, center_sec=win.center_sec, top_k=top_k)
            )

    return scores


def run_encoder_pass(
    video_path: Path,
    audio_16k: np.ndarray,
    video_frames: np.ndarray,
    *,
    window_sec: float = 1.0,
    hop_sec: float = 0.5,
    duration_sec: float,
    top_k: int = 8,
    full_scan: bool = False,
    proposal_windows: list[ProposalWindow] | None = None,
    timeline_hz: int = 100,
) -> list[EncoderScore]:
    \"\"\"
    Classify with frozen AST + ViViT.

    Default: proposal windows only. Set full_scan=True for legacy timeline sweep.
    \"\"\"
    if not full_scan and proposal_windows is not None:
        return run_encoder_on_windows(
            proposal_windows,
            video_path,
            audio_16k,
            video_frames,
            timeline_hz=timeline_hz,
            window_sec=window_sec,
            top_k=top_k,
        )

    scores: list[EncoderScore] = []
    t = 0.0
    while t < duration_sec:
        end = min(duration_sec, t + window_sec)
        scores.extend(classify_audio_window(audio_16k, t, end, top_k=top_k))

        bin_start = int(t * 100)
        bin_end = int(end * 100)
        bin_end = min(bin_end, len(video_frames))
        if bin_end > bin_start:
            clip = video_frames[bin_start:bin_end]
            scores.extend(
                classify_video_frames(clip, center_sec=0.5 * (t + end), top_k=top_k)
            )
        else:
            scores.extend(
                classify_video_clip_file(video_path, center_sec=0.5 * (t + end), top_k=top_k)
            )
        t += hop_sec
    return scores
""",
    "context/event_aggregation.py": """\"\"\"Merge symbolic tokens into event primitives with start/peak/end.\"\"\"

from __future__ import annotations

from dataclasses import dataclass, field

from haptic_gt.context.encoders import EncoderScore
from haptic_gt.context.context_detectors import SymbolicToken
from haptic_gt.context.taxonomy import Taxonomy, load_taxonomy, match_label_to_category


@dataclass
class EventPrimitive:
    category: str | None
    label: str
    start_sec: float
    peak_sec: float
    end_sec: float
    confidence: float
    sources: list[str] = field(default_factory=list)
    context_token: bool = False
    audio_score: float | None = None
    video_score: float | None = None


def _merge_tokens(
    items: list[SymbolicToken],
    taxonomy: Taxonomy,
) -> list[EventPrimitive]:
    \"\"\"Peak-split impulsive tokens; gap-merge sustained ones.\"\"\"
    if not items:
        return []

    by_category: dict[str, list[SymbolicToken]] = {}
    for tok in items:
        cat = tok.category or "unknown"
        by_category.setdefault(cat, []).append(tok)

    primitives: list[EventPrimitive] = []
    for cat_name, group in by_category.items():
        cat_cfg = taxonomy.categories.get(cat_name)
        if cat_cfg and cat_cfg.impulsive:
            scores = [
                EncoderScore(
                    time_sec=t.time_sec,
                    label=t.label,
                    score=t.confidence,
                    source=t.modality,
                )
                for t in group
            ]
            primitives.extend(
                _peak_split_scores(
                    scores,
                    category=cat_name,
                    taxonomy=taxonomy,
                    context_token=True,
                )
            )
            continue

        sorted_items = sorted(group, key=lambda x: x.time_sec)
        clusters: list[list[SymbolicToken]] = [[sorted_items[0]]]
        for tok in sorted_items[1:]:
            if tok.time_sec - clusters[-1][-1].time_sec <= 0.75:
                clusters[-1].append(tok)
            else:
                clusters.append([tok])
        for cluster in clusters:
            peak_tok = max(cluster, key=lambda x: x.confidence)
            half_win = 0.25
            primitives.append(
                EventPrimitive(
                    category=cat_name if cat_name != "unknown" else None,
                    label=peak_tok.label,
                    start_sec=max(0.0, cluster[0].time_sec - half_win),
                    peak_sec=peak_tok.time_sec,
                    end_sec=cluster[-1].time_sec + half_win,
                    confidence=peak_tok.confidence,
                    sources=sorted({g.modality for g in cluster}),
                    context_token=True,
                )
            )
    return primitives


def _local_peaks(
    times: list[float],
    scores: list[float],
    *,
    min_score: float,
    min_distance_sec: float,
) -> list[int]:
    \"\"\"Indices of local maxima separated by at least min_distance_sec.\"\"\"
    if not times:
        return []

    # Collapse duplicate timestamps to max score
    buckets: dict[float, float] = {}
    for t, s in zip(times, scores):
        buckets[t] = max(buckets.get(t, 0.0), s)
    ordered_t = sorted(buckets)
    ordered_s = [buckets[t] for t in ordered_t]

    candidates: list[int] = []
    n = len(ordered_t)
    for i in range(n):
        if ordered_s[i] < min_score:
            continue
        left = ordered_s[i - 1] if i > 0 else -1.0
        right = ordered_s[i + 1] if i + 1 < n else -1.0
        if ordered_s[i] >= left and ordered_s[i] >= right:
            candidates.append(i)

    # Greedy keep highest peaks first, enforce spacing
    candidates.sort(key=lambda i: ordered_s[i], reverse=True)
    kept: list[int] = []
    for i in candidates:
        if all(abs(ordered_t[i] - ordered_t[j]) >= min_distance_sec for j in kept):
            kept.append(i)
    kept.sort()
    return kept


def _peak_split_scores(
    scores: list[EncoderScore],
    *,
    category: str,
    taxonomy: Taxonomy,
    context_token: bool = False,
) -> list[EventPrimitive]:
    \"\"\"One short event per local score peak (for gunshots / explosions).\"\"\"
    if not scores:
        return []

    min_score = taxonomy.impulsive_encoder_threshold
    half = taxonomy.impulsive_event_half_width_sec
    min_dist = taxonomy.impulsive_min_peak_distance_sec

    times = [s.time_sec for s in scores]
    vals = [s.score for s in scores]
    peak_idxs = _local_peaks(
        times, vals, min_score=min_score, min_distance_sec=min_dist
    )

    buckets: dict[float, float] = {}
    for t, s in zip(times, vals):
        buckets[t] = max(buckets.get(t, 0.0), s)
    ordered_t = sorted(buckets)

    if not peak_idxs:
        best = max(scores, key=lambda s: s.score)
        if best.score < min_score:
            return []
        peak_times = {best.time_sec}
    else:
        peak_times = {ordered_t[i] for i in peak_idxs}

    peak_scores: list[EncoderScore] = []
    for t in sorted(peak_times):
        nearby = [s for s in scores if abs(s.time_sec - t) <= 0.01]
        if not nearby:
            continue
        peak_scores.append(max(nearby, key=lambda s: s.score))

    primitives: list[EventPrimitive] = []
    for peak in peak_scores:
        nearby = [
            s
            for s in scores
            if abs(s.time_sec - peak.time_sec) <= half + 0.15
            and s.score >= min_score * 0.8
        ]
        sources = sorted({s.source for s in (nearby or [peak])})
        audio_scores = [s.score for s in nearby if s.source == "audio"] or (
            [peak.score] if peak.source == "audio" else []
        )
        video_scores = [s.score for s in nearby if s.source == "video"] or (
            [peak.score] if peak.source == "video" else []
        )
        primitives.append(
            EventPrimitive(
                category=category,
                label=peak.label,
                start_sec=max(0.0, peak.time_sec - half),
                peak_sec=peak.time_sec,
                end_sec=peak.time_sec + half,
                confidence=peak.score,
                sources=sources,
                context_token=context_token,
                audio_score=max(audio_scores) if audio_scores else None,
                video_score=max(video_scores) if video_scores else None,
            )
        )
    return primitives


def _merge_encoder_scores(
    scores: list[EncoderScore],
    taxonomy: Taxonomy,
    *,
    gap_sec: float = 1.0,
) -> list[EventPrimitive]:
    \"\"\"Aggregate encoder scores by taxonomy category.\"\"\"
    if not scores:
        return []

    by_category: dict[str, list[EncoderScore]] = {}
    for s in scores:
        cat = match_label_to_category(taxonomy, s.label, s.source)
        if cat is None:
            continue
        by_category.setdefault(cat, []).append(s)

    primitives: list[EventPrimitive] = []
    for cat_name, group in by_category.items():
        cat_cfg = taxonomy.categories.get(cat_name)
        if cat_cfg and cat_cfg.impulsive:
            primitives.extend(
                _peak_split_scores(group, category=cat_name, taxonomy=taxonomy)
            )
            continue

        # Sustained categories (vehicle, weather): gap-merge above-threshold hits
        threshold = getattr(taxonomy, "sustained_encoder_threshold", taxonomy.encoder_threshold)
        group = sorted(
            [s for s in group if s.score >= threshold],
            key=lambda x: x.time_sec,
        )
        if not group:
            continue
        clusters: list[list[EncoderScore]] = [[group[0]]]
        for item in group[1:]:
            if item.time_sec - clusters[-1][-1].time_sec <= gap_sec:
                clusters[-1].append(item)
            else:
                clusters.append([item])
        for cluster in clusters:
            peak = max(cluster, key=lambda x: x.score)
            sources = sorted({c.source for c in cluster})
            audio_scores = [c.score for c in cluster if c.source == "audio"]
            video_scores = [c.score for c in cluster if c.source == "video"]
            primitives.append(
                EventPrimitive(
                    category=cat_name,
                    label=peak.label,
                    start_sec=max(0.0, cluster[0].time_sec - 0.5),
                    peak_sec=peak.time_sec,
                    end_sec=cluster[-1].time_sec + 0.5,
                    confidence=peak.score,
                    sources=sources,
                    context_token=False,
                    audio_score=max(audio_scores) if audio_scores else None,
                    video_score=max(video_scores) if video_scores else None,
                )
            )
    return primitives


def _overlaps(a: EventPrimitive, b: EventPrimitive, *, margin_sec: float = 0.35) -> bool:
    if a.category != b.category:
        return False
    # Impulsive: only treat as same event if peaks are very close
    if abs(a.peak_sec - b.peak_sec) <= margin_sec:
        return True
    return not (a.end_sec + margin_sec < b.start_sec or b.end_sec + margin_sec < a.start_sec)


def aggregate_events(
    tokens: list[SymbolicToken],
    encoder_scores: list[EncoderScore],
    taxonomy: Taxonomy | None = None,
) -> list[EventPrimitive]:
    \"\"\"Combine context symbolic tokens and encoder scores into primitives.\"\"\"
    taxonomy = taxonomy or load_taxonomy()
    token_primitives = _merge_tokens(tokens, taxonomy)
    encoder_primitives = _merge_encoder_scores(encoder_scores, taxonomy)

    if not token_primitives:
        return encoder_primitives
    if not encoder_primitives:
        return token_primitives

    merged = list(token_primitives)
    for enc in encoder_primitives:
        if any(_overlaps(enc, tok) for tok in token_primitives):
            continue
        merged.append(enc)
    merged.sort(key=lambda p: p.start_sec)
    return merged
""",
    "context/frozen_fusion.py": """\"\"\"Phase 2 frozen cross-modal fusion (rule-based, no training).\"\"\"

from __future__ import annotations

from dataclasses import dataclass, field

from haptic_gt.context.encoders import EncoderScore
from haptic_gt.context.event_aggregation import EventPrimitive, aggregate_events
from haptic_gt.context.context_detectors import SymbolicToken
from haptic_gt.context.taxonomy import Taxonomy, load_taxonomy, match_label_to_category


@dataclass
class DetectedEvent:
    category: str
    label: str
    start_sec: float
    peak_sec: float
    end_sec: float
    confidence: float
    context_token: bool = False
    audio_score: float | None = None
    video_score: float | None = None
    sources: list[str] = field(default_factory=list)


def _best_encoder_scores(
    scores: list[EncoderScore],
    taxonomy: Taxonomy,
    center_sec: float,
    tolerance_sec: float = 0.75,
) -> dict[str, tuple[str, float, str]]:
    \"\"\"Return best audio/video score per category near center_sec.\"\"\"
    best: dict[str, tuple[str, float, str]] = {}
    for s in scores:
        if abs(s.time_sec - center_sec) > tolerance_sec:
            continue
        cat = match_label_to_category(taxonomy, s.label, s.source)
        if cat is None:
            continue
        prev = best.get(cat)
        if prev is None or s.score > prev[1]:
            best[cat] = (s.label, s.score, s.source)
    return best


def fuse_events(
    tokens: list[SymbolicToken],
    encoder_scores: list[EncoderScore],
    taxonomy: Taxonomy | None = None,
) -> list[DetectedEvent]:
    \"\"\"
    Frozen cross-modal fusion at event level.

    1. Aggregate tokens/scores into primitives
    2. Map labels to taxonomy categories
    3. Apply per-category fusion rules and thresholds
    \"\"\"
    taxonomy = taxonomy or load_taxonomy()
    primitives = aggregate_events(tokens, encoder_scores, taxonomy)
    detected: list[DetectedEvent] = []

    for prim in primitives:
        cat_name = prim.category
        if cat_name is None:
            cat_name = match_label_to_category(taxonomy, prim.label, "audio")
        if cat_name is None:
            cat_name = match_label_to_category(taxonomy, prim.label, "video")
        if cat_name is None or cat_name not in taxonomy.categories:
            continue

        cat_cfg = taxonomy.categories[cat_name]
        nearby = _best_encoder_scores(encoder_scores, taxonomy, prim.peak_sec)
        audio_score = prim.audio_score
        video_score = prim.video_score
        if cat_name in nearby:
            lbl, sc, src = nearby[cat_name]
            if src == "audio":
                audio_score = max(audio_score or 0.0, sc)
            else:
                video_score = max(video_score or 0.0, sc)
            if prim.label == lbl or prim.label in lbl:
                pass
            elif audio_score is None and video_score is None:
                prim.label = lbl

        has_context = prim.context_token
        if cat_cfg.impulsive:
            cat_threshold = taxonomy.impulsive_encoder_threshold
        elif hasattr(taxonomy, "sustained_encoder_threshold"):
            cat_threshold = taxonomy.sustained_encoder_threshold
        else:
            cat_threshold = taxonomy.encoder_threshold
        has_audio = (audio_score or 0.0) >= cat_threshold
        has_video = (video_score or 0.0) >= cat_threshold

        if cat_cfg.require_context_or_both:
            passes = has_context or (has_audio and has_video)
            if not passes and has_audio and cat_cfg.impulsive:
                passes = (audio_score or 0.0) >= cat_threshold
            if not passes:
                continue
        else:
            weighted = (
                cat_cfg.audio_weight * (audio_score or 0.0)
                + cat_cfg.video_weight * (video_score or 0.0)
            )
            score_floor = max(prim.confidence, weighted)
            if not has_context and score_floor < cat_threshold:
                continue

        if has_context and (has_audio or has_video):
            prim.confidence = min(1.0, prim.confidence * 1.05)

        detected.append(
            DetectedEvent(
                category=cat_name,
                label=prim.label,
                start_sec=prim.start_sec,
                peak_sec=prim.peak_sec,
                end_sec=prim.end_sec,
                confidence=prim.confidence,
                context_token=prim.context_token,
                audio_score=audio_score,
                video_score=video_score,
                sources=prim.sources,
            )
        )

    # Deduplicate near-duplicate peaks only (keep distinct shots/blasts)
    detected.sort(key=lambda e: (e.category, e.peak_sec, -e.confidence))
    merged: list[DetectedEvent] = []
    for ev in detected:
        if merged and ev.category == merged[-1].category:
            same_peak = abs(ev.peak_sec - merged[-1].peak_sec) <= 0.35
            if same_peak:
                if ev.confidence > merged[-1].confidence:
                    merged[-1] = ev
                continue
        merged.append(ev)

    merged.sort(key=lambda e: e.start_sec)
    return merged
""",
    "context/mask.py": """\"\"\"Build gated audio from detected event segments.\"\"\"

from __future__ import annotations

from pathlib import Path

import numpy as np
import soundfile as sf

from haptic_gt.audio_io import INPUT_SR
from haptic_gt.context.frozen_fusion import DetectedEvent
from haptic_gt.context.taxonomy import Taxonomy, load_taxonomy


def resolve_gate_categories(
    taxonomy: Taxonomy,
    gate_categories: list[str] | None = None,
) -> list[str]:
    \"\"\"Return categories used for haptic gating.\"\"\"
    if gate_categories is not None:
        return list(gate_categories)
    return [
        name
        for name, cat in taxonomy.categories.items()
        if cat.include_in_haptic_gate
    ]


def events_for_haptic_gate(
    events: list[DetectedEvent],
    taxonomy: Taxonomy,
    gate_categories: list[str] | None = None,
) -> list[DetectedEvent]:
    \"\"\"Events that should contribute to gated_audio.wav.\"\"\"
    allowed = set(resolve_gate_categories(taxonomy, gate_categories))
    gated: list[DetectedEvent] = []
    for ev in events:
        if ev.category in allowed:
            gated.append(ev)
    return gated


def event_included_in_gate(
    event: DetectedEvent,
    taxonomy: Taxonomy,
    gate_categories: list[str] | None = None,
) -> bool:
    allowed = set(resolve_gate_categories(taxonomy, gate_categories))
    return event.category in allowed


def build_event_mask(
    duration_samples: int,
    sample_rate: int,
    events: list[DetectedEvent],
    *,
    fade_ms: float = 10.0,
) -> np.ndarray:
    \"\"\"Binary mask with short linear fades at segment edges.\"\"\"
    mask = np.zeros(duration_samples, dtype=np.float32)
    fade = max(1, int(sample_rate * fade_ms / 1000.0))

    for ev in events:
        s0 = max(0, int(ev.start_sec * sample_rate))
        s1 = min(duration_samples, int(ev.end_sec * sample_rate))
        if s1 <= s0:
            continue
        mask[s0:s1] = 1.0
        f0 = min(fade, (s1 - s0) // 2)
        if f0 > 0:
            ramp = np.linspace(0.0, 1.0, f0, dtype=np.float32)
            mask[s0 : s0 + f0] = np.maximum(mask[s0 : s0 + f0], ramp)
            mask[s1 - f0 : s1] = np.maximum(mask[s1 - f0 : s1], ramp[::-1])

    return mask


def apply_gate(
    source_wav: str | Path,
    output_wav: str | Path,
    events: list[DetectedEvent],
    *,
    sample_rate: int = INPUT_SR,
    taxonomy: Taxonomy | None = None,
    gate_categories: list[str] | None = None,
) -> Path:
    \"\"\"Write gated mono WAV keeping only selected event segments.\"\"\"
    source_wav = Path(source_wav)
    output_wav = Path(output_wav)
    taxonomy = taxonomy or load_taxonomy()
    gate_events = events_for_haptic_gate(events, taxonomy, gate_categories=gate_categories)

    audio, sr = sf.read(source_wav, always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    audio = audio.astype(np.float32)

    if sr != sample_rate:
        import librosa

        audio = librosa.resample(audio, orig_sr=sr, target_sr=sample_rate)
        sr = sample_rate

    mask = build_event_mask(len(audio), sr, gate_events)
    gated = audio * mask

    output_wav.parent.mkdir(parents=True, exist_ok=True)
    sf.write(output_wav, np.clip(gated, -1.0, 1.0), sr, subtype="PCM_16")
    return output_wav
""",
    "context/onset_refine.py": """\"\"\"Refine event peak/start/end using spectral-flux onset alignment.\"\"\"

from __future__ import annotations

from pathlib import Path

import numpy as np
import soundfile as sf

from haptic_gt.context.frozen_fusion import DetectedEvent
from haptic_gt.context.taxonomy import Taxonomy, load_taxonomy


def _envelope_rms(
    audio: np.ndarray,
    sr: int,
    *,
    hop_ms: float = 5.0,
) -> tuple[np.ndarray, np.ndarray]:
    hop = max(1, int(sr * hop_ms / 1000))
    n_frames = max(1, (len(audio) + hop - 1) // hop)
    env: list[float] = []
    times: list[float] = []
    for i in range(n_frames):
        s0 = i * hop
        s1 = min(len(audio), s0 + hop)
        chunk = audio[s0:s1]
        env.append(float(np.sqrt(np.mean(chunk**2) + 1e-12)))
        times.append((s0 + s1) / 2.0 / sr)
    return np.asarray(times, dtype=np.float64), np.asarray(env, dtype=np.float64)


def _spectral_flux(
    audio: np.ndarray,
    sr: int,
    *,
    hop_ms: float = 5.0,
    n_fft: int = 512,
    hf_weight: float = 1.5,
) -> tuple[np.ndarray, np.ndarray]:
    \"\"\"
    Positive spectral flux with high-frequency emphasis.

    Better than RMS for impulsive onsets (gunshot / explosion attacks).
    \"\"\"
    hop = max(1, int(sr * hop_ms / 1000))
    if len(audio) < n_fft:
        audio = np.pad(audio, (0, n_fft - len(audio)))

    window = np.hanning(n_fft).astype(np.float32)
    n_frames = 1 + max(0, (len(audio) - n_fft) // hop)
    if n_frames < 2:
        return np.array([0.0]), np.array([0.0])

    freqs = np.fft.rfftfreq(n_fft, d=1.0 / sr)
    # Emphasize mid/high bands typical of blast attacks
    band_w = np.ones_like(freqs, dtype=np.float32)
    band_w[freqs >= 500.0] = hf_weight
    band_w[freqs >= 2000.0] = hf_weight * 1.25
    band_w[freqs < 80.0] = 0.35

    mags: list[np.ndarray] = []
    times: list[float] = []
    for i in range(n_frames):
        s0 = i * hop
        s1 = s0 + n_fft
        if s1 > len(audio):
            break
        frame = audio[s0:s1] * window
        mag = np.abs(np.fft.rfft(frame)).astype(np.float32) * band_w
        mags.append(mag)
        times.append((s0 + n_fft / 2.0) / sr)

    if len(mags) < 2:
        return np.asarray(times, dtype=np.float64), np.zeros(len(times), dtype=np.float64)

    flux = np.zeros(len(mags), dtype=np.float64)
    for i in range(1, len(mags)):
        diff = mags[i] - mags[i - 1]
        flux[i] = float(np.sum(np.maximum(diff, 0.0)))

    return np.asarray(times, dtype=np.float64), flux


def _pick_onset_from_flux(
    times: np.ndarray,
    flux: np.ndarray,
    *,
    min_ratio: float = 0.45,
) -> float | None:
    \"\"\"Pick the strongest onset; break ties toward later peaks.\"\"\"
    if flux.size == 0:
        return None
    peak = float(np.max(flux))
    if peak < 1e-12:
        return None

    floor = peak * min_ratio
    candidates: list[int] = []
    for i in range(1, len(flux) - 1):
        if flux[i] < floor:
            continue
        if flux[i] >= flux[i - 1] and flux[i] >= flux[i + 1]:
            candidates.append(i)
    if not candidates:
        return float(times[int(np.argmax(flux))])

    # Prefer the highest flux; among near-ties, pick later (main blast after precursor)
    best = candidates[0]
    best_val = flux[best]
    for idx in candidates[1:]:
        if flux[idx] > best_val * 1.05:
            best = idx
            best_val = flux[idx]
        elif flux[idx] >= best_val * 0.92 and times[idx] > times[best]:
            best = idx
            best_val = flux[idx]
    return float(times[best])


def _find_acoustic_peak(
    audio: np.ndarray,
    sr: int,
    center_sec: float,
    radius_sec: float,
) -> float:
    \"\"\"Peak RMS sample time near the classifier's rough center.\"\"\"
    s0 = max(0, int((center_sec - radius_sec) * sr))
    s1 = min(len(audio), int((center_sec + radius_sec) * sr))
    if s1 <= s0:
        return center_sec

    seg = audio[s0:s1]
    times, env = _envelope_rms(seg, sr)
    if env.size == 0:
        return center_sec
    peak_idx = int(np.argmax(env))
    return float(s0 / sr + times[peak_idx])


def _find_impulsive_peak(
    audio: np.ndarray,
    sr: int,
    center_sec: float,
    taxonomy: Taxonomy,
) -> float:
    \"\"\"
    Snap impulsive events to spectral-flux onset near the classifier hint.

    Uses high-frequency-weighted spectral flux (not RMS max), which better
    matches gunshot / explosion attacks. Falls back to RMS if flux is weak.
    \"\"\"
    back = taxonomy.impulsive_onset_back_sec
    forward = taxonomy.impulsive_onset_forward_sec
    hop_ms = taxonomy.onset_flux_hop_ms
    s0 = max(0, int((center_sec - back) * sr))
    s1 = min(len(audio), int((center_sec + forward) * sr))
    if s1 <= s0:
        return center_sec

    seg = audio[s0:s1]
    times, flux = _spectral_flux(seg, sr, hop_ms=hop_ms)
    onset_rel = _pick_onset_from_flux(times, flux, min_ratio=taxonomy.onset_flux_min_ratio)
    if onset_rel is not None:
        return float(s0 / sr + onset_rel)

    # Fallback: RMS peak in the same window
    times_rms, env = _envelope_rms(seg, sr, hop_ms=max(2.5, hop_ms))
    if env.size == 0:
        return center_sec
    return float(s0 / sr + times_rms[int(np.argmax(env))])


def _extend_decay_tail(
    audio: np.ndarray,
    sr: int,
    peak_sec: float,
    *,
    threshold_ratio: float,
    max_tail_sec: float,
    duration_sec: float,
) -> float:
    \"\"\"Extend gate end while RMS stays above a fraction of the peak.\"\"\"
    peak_idx = int(peak_sec * sr)
    hop = max(1, int(sr * 0.005))
    lo = max(0, peak_idx - hop)
    hi = min(len(audio), peak_idx + hop)
    peak_val = float(np.sqrt(np.mean(audio[lo:hi] ** 2) + 1e-12))
    if peak_val < 1e-8:
        return peak_sec

    thr = peak_val * threshold_ratio
    end_idx = min(len(audio), int((peak_sec + max_tail_sec) * sr))
    i = peak_idx
    while i < end_idx:
        s1 = min(len(audio), i + hop)
        v = float(np.sqrt(np.mean(audio[i:s1] ** 2) + 1e-12))
        if v < thr:
            break
        i += hop
    return min(duration_sec, i / sr)


def refine_event_timing(
    events: list[DetectedEvent],
    source_wav: str | Path,
    taxonomy: Taxonomy | None = None,
) -> list[DetectedEvent]:
    \"\"\"
    Snap classifier window centers to acoustic onsets and fix gate spans.

    Impulsive: spectral-flux onset; short pre-roll + decay tail.
    Sustained: RMS peak; span capped for gating metadata.
    \"\"\"
    taxonomy = taxonomy or load_taxonomy()
    source_wav = Path(source_wav)
    audio, sr = sf.read(source_wav, always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    audio = audio.astype(np.float32)
    duration_sec = len(audio) / sr

    radius = taxonomy.impulsive_onset_search_radius_sec
    half = taxonomy.impulsive_event_half_width_sec
    pre_roll = taxonomy.impulsive_pre_roll_sec
    max_tail = taxonomy.impulsive_decay_tail_sec
    decay_thr = taxonomy.impulsive_decay_threshold
    sustained_max = taxonomy.sustained_max_gate_sec

    refined: list[DetectedEvent] = []
    for ev in events:
        cat_cfg = taxonomy.categories.get(ev.category)
        impulsive = bool(cat_cfg and cat_cfg.impulsive)

        if impulsive:
            acoustic_peak = _find_impulsive_peak(audio, sr, ev.peak_sec, taxonomy)
        else:
            acoustic_peak = _find_acoustic_peak(audio, sr, ev.peak_sec, radius)

        if impulsive:
            peak = acoustic_peak
            start = max(0.0, peak - pre_roll)
            # Keep some post-onset content for algorithm context
            min_end = peak + half
            decay_end = _extend_decay_tail(
                audio,
                sr,
                peak,
                threshold_ratio=decay_thr,
                max_tail_sec=max_tail,
                duration_sec=duration_sec,
            )
            end = min(duration_sec, max(min_end, decay_end))
        else:
            peak = acoustic_peak
            span = ev.end_sec - ev.start_sec
            if span > sustained_max:
                start = max(0.0, peak - sustained_max / 2.0)
                end = min(duration_sec, start + sustained_max)
            else:
                start = max(0.0, ev.start_sec)
                end = min(duration_sec, ev.end_sec)
            # Keep peak inside [start, end]
            peak = min(max(peak, start), end)

        refined.append(
            DetectedEvent(
                category=ev.category,
                label=ev.label,
                start_sec=start,
                peak_sec=peak,
                end_sec=end,
                confidence=ev.confidence,
                context_token=ev.context_token,
                audio_score=ev.audio_score,
                video_score=ev.video_score,
                sources=list(ev.sources),
            )
        )
    return refined
""",
    "context/proposals.py": """\"\"\"Cheap RMS onset proposals before frozen transformer classification.\"\"\"

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import soundfile as sf

from haptic_gt.audio_io import INPUT_SR
from haptic_gt.context.taxonomy import Taxonomy, load_taxonomy


@dataclass
class ProposalWindow:
    center_sec: float
    start_sec: float
    end_sec: float
    rms_score: float


def _envelope_rms(
    audio: np.ndarray,
    sr: int,
    *,
    hop_ms: float,
) -> tuple[np.ndarray, np.ndarray]:
    hop = max(1, int(sr * hop_ms / 1000))
    n_frames = max(1, (len(audio) + hop - 1) // hop)
    env: list[float] = []
    times: list[float] = []
    for i in range(n_frames):
        s0 = i * hop
        s1 = min(len(audio), s0 + hop)
        chunk = audio[s0:s1]
        env.append(float(np.sqrt(np.mean(chunk**2) + 1e-12)))
        times.append((s0 + s1) / 2.0 / sr)
    return np.asarray(times, dtype=np.float64), np.asarray(env, dtype=np.float64)


def _local_peak_indices(
    values: np.ndarray,
    *,
    min_score: float,
    min_distance_frames: int,
) -> list[int]:
    candidates: list[int] = []
    n = len(values)
    for i in range(n):
        if values[i] < min_score:
            continue
        left = values[i - 1] if i > 0 else -1.0
        right = values[i + 1] if i + 1 < n else -1.0
        if values[i] >= left and values[i] >= right:
            candidates.append(i)

    candidates.sort(key=lambda i: values[i], reverse=True)
    kept: list[int] = []
    for i in candidates:
        if all(abs(i - j) >= min_distance_frames for j in kept):
            kept.append(i)
    kept.sort()
    return kept


def merge_proposal_windows(
    windows: list[ProposalWindow],
    *,
    merge_gap_sec: float = 0.35,
) -> list[ProposalWindow]:
    \"\"\"Merge overlapping or nearly-adjacent proposal windows.\"\"\"
    if not windows:
        return []

    ordered = sorted(windows, key=lambda w: w.center_sec)
    merged: list[ProposalWindow] = [ordered[0]]
    for win in ordered[1:]:
        prev = merged[-1]
        if win.start_sec <= prev.end_sec + merge_gap_sec:
            merged[-1] = ProposalWindow(
                center_sec=win.center_sec
                if win.rms_score > prev.rms_score
                else prev.center_sec,
                start_sec=min(prev.start_sec, win.start_sec),
                end_sec=max(prev.end_sec, win.end_sec),
                rms_score=max(prev.rms_score, win.rms_score),
            )
        else:
            merged.append(win)
    return merged


def propose_onsets(
    source_wav: str | Path,
    taxonomy: Taxonomy | None = None,
    *,
    sample_rate: int = INPUT_SR,
) -> list[ProposalWindow]:
    \"\"\"
    Find candidate transient times via short-hop RMS local maxima.

    Uses adaptive thresholding relative to the clip's RMS envelope.
    \"\"\"
    taxonomy = taxonomy or load_taxonomy()
    source_wav = Path(source_wav)

    audio, sr = sf.read(source_wav, always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    audio = audio.astype(np.float32)
    if sr != sample_rate:
        import librosa

        audio = librosa.resample(audio, orig_sr=sr, target_sr=sample_rate)
        sr = sample_rate

    if len(audio) == 0:
        return []

    hop_ms = taxonomy.proposal_rms_hop_ms
    pad_sec = taxonomy.proposal_search_pad_sec
    min_dist_sec = taxonomy.impulsive_min_peak_distance_sec

    times, env = _envelope_rms(audio, sr, hop_ms=hop_ms)
    if env.size == 0:
        return []

    peak_val = float(np.max(env))
    if peak_val < 1e-8:
        return []

    threshold = peak_val * taxonomy.proposal_threshold_ratio
    hop_sec = hop_ms / 1000.0
    min_dist_frames = max(1, int(round(min_dist_sec / hop_sec)))

    peak_idxs = _local_peak_indices(
        env,
        min_score=threshold,
        min_distance_frames=min_dist_frames,
    )

    duration_sec = len(audio) / sr
    windows: list[ProposalWindow] = []
    for idx in peak_idxs:
        center = float(times[idx])
        windows.append(
            ProposalWindow(
                center_sec=center,
                start_sec=max(0.0, center - pad_sec),
                end_sec=min(duration_sec, center + pad_sec),
                rms_score=float(env[idx]),
            )
        )

    return merge_proposal_windows(windows)


def propose_sustained_scan(
    duration_sec: float,
    taxonomy: Taxonomy | None = None,
    *,
    pad_sec: float | None = None,
) -> list[ProposalWindow]:
    \"\"\"Uniform timeline windows for sustained categories (engine rumble, etc.).\"\"\"
    taxonomy = taxonomy or load_taxonomy()
    hop = taxonomy.proposal_sustained_hop_sec
    pad = pad_sec if pad_sec is not None else taxonomy.proposal_search_pad_sec
    if duration_sec <= 0 or hop <= 0:
        return []

    centers: list[float] = []
    t = hop / 2.0
    while t < duration_sec:
        centers.append(t)
        t += hop

    return [
        ProposalWindow(
            center_sec=c,
            start_sec=max(0.0, c - pad),
            end_sec=min(duration_sec, c + pad),
            rms_score=0.0,
        )
        for c in centers
    ]


def _dedupe_nearby_windows(
    windows: list[ProposalWindow],
    *,
    min_center_gap_sec: float,
) -> list[ProposalWindow]:
    \"\"\"Keep highest-scoring window when centers are very close.\"\"\"
    if not windows:
        return []

    ordered = sorted(windows, key=lambda w: w.center_sec)
    kept: list[ProposalWindow] = [ordered[0]]
    for win in ordered[1:]:
        if win.center_sec - kept[-1].center_sec < min_center_gap_sec:
            if win.rms_score > kept[-1].rms_score:
                kept[-1] = win
        else:
            kept.append(win)
    return kept


def propose_all_windows(
    source_wav: str | Path,
    taxonomy: Taxonomy | None = None,
    *,
    sample_rate: int = INPUT_SR,
    include_sustained_scan: bool = True,
) -> list[ProposalWindow]:
    \"\"\"
    Onset transients plus optional sustained timeline scan for classifiers.

    Onset peaks are preserved; sustained windows fill gaps for vehicle / activity.
    \"\"\"
    taxonomy = taxonomy or load_taxonomy()
    onset = propose_onsets(source_wav, taxonomy, sample_rate=sample_rate)
    if not include_sustained_scan:
        return onset

    source_wav = Path(source_wav)
    audio, sr = sf.read(source_wav, always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    duration_sec = len(audio) / (sr if sr else sample_rate)
    sustained = propose_sustained_scan(duration_sec, taxonomy)

    combined = _dedupe_nearby_windows(
        onset + sustained,
        min_center_gap_sec=0.35,
    )
    return combined
""",
    "context/taxonomy.py": """\"\"\"Load and query the predefined event taxonomy.\"\"\"

from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path

import yaml

DEFAULT_TAXONOMY_PATH = Path(__file__).with_name("taxonomy.yaml")


@dataclass
class CategoryConfig:
    name: str
    audioset_labels: list[str] = field(default_factory=list)
    kinetics_labels: list[str] = field(default_factory=list)
    context_token_labels: list[str] = field(default_factory=list)
    audio_weight: float = 0.5
    video_weight: float = 0.5
    require_context_or_both: bool = False
    impulsive: bool = False
    include_in_haptic_gate: bool = False


@dataclass
class Taxonomy:
    timeline_hz: int = 100
    context_detector_threshold: float = 0.85
    encoder_threshold: float = 0.35
    impulsive_encoder_threshold: float = 0.30
    sustained_encoder_threshold: float = 0.25
    impulsive_min_peak_distance_sec: float = 0.7
    impulsive_event_half_width_sec: float = 0.45
    impulsive_onset_search_radius_sec: float = 0.75
    impulsive_onset_back_sec: float = 0.6
    impulsive_onset_forward_sec: float = 1.2
    impulsive_decay_tail_sec: float = 0.85
    impulsive_decay_threshold: float = 0.12
    impulsive_pre_roll_sec: float = 0.08
    sustained_max_gate_sec: float = 2.5
    proposal_rms_hop_ms: float = 5.0
    proposal_threshold_ratio: float = 0.25
    proposal_search_pad_sec: float = 0.75
    proposal_window_sec: float = 1.0
    proposal_sustained_hop_sec: float = 2.0
    onset_flux_hop_ms: float = 5.0
    onset_flux_min_ratio: float = 0.45
    categories: dict[str, CategoryConfig] = field(default_factory=dict)

    def all_audioset_labels(self) -> set[str]:
        out: set[str] = set()
        for cat in self.categories.values():
            out.update(cat.audioset_labels)
        return out

    def all_kinetics_labels(self) -> set[str]:
        out: set[str] = set()
        for cat in self.categories.values():
            out.update(cat.kinetics_labels)
        return out


def _normalize(label: str) -> str:
    return " ".join(label.lower().strip().split())


def load_taxonomy(path: str | Path | None = None) -> Taxonomy:
    path = Path(path) if path else DEFAULT_TAXONOMY_PATH
    raw = yaml.safe_load(path.read_text(encoding="utf-8"))
    categories: dict[str, CategoryConfig] = {}
    for name, cfg in raw.get("categories", {}).items():
        categories[name] = CategoryConfig(
            name=name,
            audioset_labels=list(cfg.get("audioset_labels", [])),
            kinetics_labels=list(cfg.get("kinetics_labels", [])),
            context_token_labels=list(cfg.get("context_token_labels", [])),
            audio_weight=float(cfg.get("audio_weight", 0.5)),
            video_weight=float(cfg.get("video_weight", 0.5)),
            require_context_or_both=bool(cfg.get("require_context_or_both", False)),
            impulsive=bool(cfg.get("impulsive", False)),
            include_in_haptic_gate=bool(
                cfg.get("include_in_haptic_gate", bool(cfg.get("impulsive", False)))
            ),
        )
    return Taxonomy(
        timeline_hz=int(raw.get("timeline_hz", 100)),
        context_detector_threshold=float(raw.get("context_detector_threshold", 0.85)),
        encoder_threshold=float(raw.get("encoder_threshold", 0.35)),
        impulsive_encoder_threshold=float(raw.get("impulsive_encoder_threshold", 0.30)),
        sustained_encoder_threshold=float(raw.get("sustained_encoder_threshold", 0.25)),
        impulsive_min_peak_distance_sec=float(
            raw.get("impulsive_min_peak_distance_sec", 0.7)
        ),
        impulsive_event_half_width_sec=float(
            raw.get("impulsive_event_half_width_sec", 0.45)
        ),
        impulsive_onset_search_radius_sec=float(
            raw.get("impulsive_onset_search_radius_sec", 0.75)
        ),
        impulsive_onset_back_sec=float(raw.get("impulsive_onset_back_sec", 0.6)),
        impulsive_onset_forward_sec=float(raw.get("impulsive_onset_forward_sec", 1.2)),
        impulsive_decay_tail_sec=float(raw.get("impulsive_decay_tail_sec", 0.85)),
        impulsive_decay_threshold=float(raw.get("impulsive_decay_threshold", 0.12)),
        impulsive_pre_roll_sec=float(raw.get("impulsive_pre_roll_sec", 0.08)),
        sustained_max_gate_sec=float(raw.get("sustained_max_gate_sec", 2.5)),
        proposal_rms_hop_ms=float(raw.get("proposal_rms_hop_ms", 5.0)),
        proposal_threshold_ratio=float(raw.get("proposal_threshold_ratio", 0.25)),
        proposal_search_pad_sec=float(raw.get("proposal_search_pad_sec", 0.75)),
        proposal_window_sec=float(raw.get("proposal_window_sec", 1.0)),
        proposal_sustained_hop_sec=float(raw.get("proposal_sustained_hop_sec", 2.0)),
        onset_flux_hop_ms=float(raw.get("onset_flux_hop_ms", 5.0)),
        onset_flux_min_ratio=float(raw.get("onset_flux_min_ratio", 0.45)),
        categories=categories,
    )


def match_label_to_category(taxonomy: Taxonomy, label: str, source: str) -> str | None:
    \"\"\"Map a raw model label to a taxonomy category name, or None.\"\"\"
    norm = _normalize(label)
    for cat_name, cat in taxonomy.categories.items():
        if source in ("audio", "context") or source == "audioset":
            pool = cat.audioset_labels + cat.context_token_labels
        elif source in ("video", "kinetics"):
            pool = cat.kinetics_labels
        elif source == "context_token":
            pool = cat.context_token_labels
        else:
            pool = (
                cat.audioset_labels
                + cat.kinetics_labels
                + cat.context_token_labels
            )
        for candidate in pool:
            c_norm = _normalize(candidate)
            if norm == c_norm or c_norm in norm or norm in c_norm:
                return cat_name
    return None
""",
    "context/tokenization.py": """\"\"\"Phase 1 tokenization: 100Hz timeline and 16kHz audio.\"\"\"

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf

MEL_SR = 16_000
TIMELINE_HZ = 100
FRAME_SIZE = 224


@dataclass
class TimelineTokens:
    \"\"\"Aligned multimodal tokens on a fixed-rate timeline.\"\"\"

    duration_sec: float
    timeline_hz: int
    video_frames: np.ndarray  # [T, 3, H, W] float32 in [0, 1]
    audio_16k: np.ndarray  # 1D float32 @ 16kHz
    source_fps: float


def _decode_video_frames(video_path: Path, n_bins: int, duration_sec: float) -> tuple[np.ndarray, float]:
    try:
        import cv2
    except ImportError as exc:
        raise RuntimeError(
            "opencv-python is required for video tokenization. "
            "Install with: pip install opencv-python"
        ) from exc

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    raw_frames: list[np.ndarray] = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        resized = cv2.resize(rgb, (FRAME_SIZE, FRAME_SIZE), interpolation=cv2.INTER_AREA)
        raw_frames.append(resized.astype(np.float32) / 255.0)

    cap.release()
    if not raw_frames:
        raise RuntimeError(f"No frames decoded from video: {video_path}")

    if duration_sec <= 0:
        duration_sec = len(raw_frames) / fps

    aligned = np.zeros((n_bins, FRAME_SIZE, FRAME_SIZE, 3), dtype=np.float32)
    for i in range(n_bins):
        t = i / TIMELINE_HZ
        src_idx = int(round(t * fps))
        src_idx = min(max(src_idx, 0), len(raw_frames) - 1)
        aligned[i] = raw_frames[src_idx]

    video = np.transpose(aligned, (0, 3, 1, 2))
    return video, float(fps)


def _load_audio_16k(source_wav: Path) -> np.ndarray:
    audio, sr = sf.read(source_wav, always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    audio = audio.astype(np.float32)
    if sr != MEL_SR:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=MEL_SR)
    return audio


def tokenize_video_audio(
    video_path: str | Path,
    source_wav: str | Path,
    *,
    timeline_hz: int = TIMELINE_HZ,
) -> TimelineTokens:
    \"\"\"Build 100Hz-aligned video frames and 16kHz mono audio.\"\"\"
    video_path = Path(video_path)
    source_wav = Path(source_wav)
    audio_16k = _load_audio_16k(source_wav)
    duration_sec = len(audio_16k) / MEL_SR
    n_bins = max(1, int(round(duration_sec * timeline_hz)))

    video_frames, source_fps = _decode_video_frames(video_path, n_bins, duration_sec)

    return TimelineTokens(
        duration_sec=duration_sec,
        timeline_hz=timeline_hz,
        video_frames=video_frames,
        audio_16k=audio_16k,
        source_fps=source_fps,
    )
""",
    "haptic_synthesis.py": """\"\"\"Stitch per-event haptic segments onto a full-length timeline.\"\"\"

from __future__ import annotations

import tempfile
from collections.abc import Callable
from pathlib import Path

import numpy as np
import soundfile as sf

from haptic_gt.audio_io import INPUT_SR, VIB_SR
from haptic_gt.context.frozen_fusion import DetectedEvent


def _read_mono(path: Path) -> tuple[np.ndarray, int]:
    audio, sr = sf.read(path, always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    return audio.astype(np.float32), sr


def _extract_clip_wav(
    source_wav: Path,
    start_sec: float,
    end_sec: float,
    dest: Path,
    *,
    sample_rate: int = INPUT_SR,
) -> None:
    audio, sr = _read_mono(source_wav)
    if sr != sample_rate:
        import librosa

        audio = librosa.resample(audio, orig_sr=sr, target_sr=sample_rate)
        sr = sample_rate

    s0 = max(0, int(start_sec * sr))
    s1 = min(len(audio), int(end_sec * sr))
    clip = audio[s0:s1]
    dest.parent.mkdir(parents=True, exist_ok=True)
    sf.write(dest, clip, sr, subtype="PCM_16")


def _event_clip_bounds(
    ev: DetectedEvent,
    duration_sec: float,
    *,
    min_clip_sec: float = 0.6,
) -> tuple[float, float]:
    \"\"\"Bounds that include the onset peak with enough post-event context.\"\"\"
    start = max(0.0, min(ev.start_sec, ev.peak_sec - 0.05))
    end = max(ev.end_sec, ev.peak_sec + 0.35)
    if end - start < min_clip_sec:
        end = min(duration_sec, start + min_clip_sec)
        start = max(0.0, end - min_clip_sec)
    end = min(duration_sec, end)
    return start, end


def _haptic_onset_sec(segment: np.ndarray, sample_rate: int, *, ratio: float = 0.18) -> float:
    \"\"\"First time the haptic |signal| crosses a fraction of its peak (attack).\"\"\"
    if segment.size == 0:
        return 0.0
    abs_seg = np.abs(segment)
    peak = float(np.max(abs_seg))
    if peak < 1e-10:
        return 0.0
    thr = peak * ratio
    hits = np.where(abs_seg >= thr)[0]
    if hits.size == 0:
        return float(np.argmax(abs_seg) / sample_rate)
    return float(hits[0] / sample_rate)


def _normalize_segment(segment: np.ndarray, *, target: float = 0.85) -> np.ndarray:
    \"\"\"Scale segment so its peak lands at `target` (haptic actuator headroom).\"\"\"
    peak = float(np.max(np.abs(segment)))
    if peak < 1e-10:
        return segment
    return segment * (target / peak)


def stitch_algorithm_output(
    source_wav: Path,
    events: list[DetectedEvent],
    output_path: Path,
    process_file: Callable[..., object],
    *,
    input_sr: int = INPUT_SR,
    output_sr: int = VIB_SR,
    process_kwargs: dict | None = None,
) -> None:
    \"\"\"
    Run an algorithm per event clip and mix results onto a full-length timeline.

    Each event's haptic segment is individually normalized to a consistent
    amplitude target before being placed, so weak events are not drowned out
    by louder ones. The haptic attack onset is aligned to the event's peak_sec.
    \"\"\"
    process_kwargs = process_kwargs or {}
    audio, sr = _read_mono(source_wav)
    if sr != input_sr:
        import librosa

        audio = librosa.resample(audio, orig_sr=sr, target_sr=input_sr)
        sr = input_sr

    duration_sec = len(audio) / sr
    total_samples = max(1, int(round(duration_sec * output_sr)))
    timeline = np.zeros(total_samples, dtype=np.float32)

    if not events:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        sf.write(output_path, timeline, output_sr, subtype="PCM_16")
        return

    with tempfile.TemporaryDirectory() as tmp:
        tmp_path = Path(tmp)
        for i, ev in enumerate(events):
            clip_in = tmp_path / f"clip_{i}.wav"
            clip_out = tmp_path / f"out_{i}.wav"
            clip_start, clip_end = _event_clip_bounds(ev, duration_sec)
            _extract_clip_wav(
                source_wav,
                clip_start,
                clip_end,
                clip_in,
                sample_rate=input_sr,
            )
            process_file(clip_in, clip_out, **process_kwargs)

            segment, seg_sr = _read_mono(clip_out)
            if seg_sr != output_sr:
                import librosa

                segment = librosa.resample(segment, orig_sr=seg_sr, target_sr=output_sr)

            # Normalize each event independently so every event is felt,
            # regardless of its raw amplitude relative to the loudest event.
            segment = _normalize_segment(segment)

            onset_sec = _haptic_onset_sec(segment, output_sr)
            place_start_sec = ev.peak_sec - onset_sec
            start_idx = int(round(place_start_sec * output_sr))
            if start_idx < 0:
                segment = segment[-start_idx:]
                start_idx = 0
            end_idx = min(total_samples, start_idx + len(segment))
            seg_len = end_idx - start_idx
            if seg_len > 0:
                timeline[start_idx:end_idx] += segment[:seg_len]

    # Clamp overlapping events without destroying individual event amplitudes
    np.clip(timeline, -1.0, 1.0, out=timeline)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(output_path, timeline, output_sr, subtype="PCM_16")
""",
    "pipeline.py": """\"\"\"End-to-end pipeline aligned with Sound2Hap signal processing.\"\"\"

from __future__ import annotations

import json
from dataclasses import dataclass, field
from pathlib import Path

from haptic_gt.algorithms import freq_shift, haptic_gen, percept, pitch_match
from haptic_gt.audio_io import INPUT_SR, VIB_SR, extract_audio_from_video, prepare_source_wav
from haptic_gt.context import detect_events
from haptic_gt.context.frozen_fusion import DetectedEvent
from haptic_gt.context.mask import events_for_haptic_gate, resolve_gate_categories
from haptic_gt.context.taxonomy import load_taxonomy
from haptic_gt.haptic_synthesis import stitch_algorithm_output

OUTPUT_NAMES = {
    "source_audio": "source_audio.wav",
    "gated_audio": "gated_audio.wav",
    "events_json": "events.json",
    "algorithm_a_perception_mapping": "algorithm_a_perception_mapping.wav",
    "algorithm_b_frequency_shifting": "algorithm_b_frequency_shifting.wav",
    "algorithm_c_pitch_matching": "algorithm_c_pitch_matching.wav",
    "algorithm_d_haptic_gen": "algorithm_d_haptic_gen.wav",
}


@dataclass
class CandidateTracks:
    \"\"\"Paths to source audio and four Sound2Hap candidate haptic tracks.\"\"\"

    source_wav: Path
    algorithm_a: Path | None
    algorithm_b: Path | None
    algorithm_c: Path | None
    algorithm_d: Path | None
    output_dir: Path
    haptic_input_wav: Path | None
    input_sample_rate: int = INPUT_SR
    output_sample_rate: int = VIB_SR
    pitch_match_info: dict | None = None
    events: list[DetectedEvent] | None = None
    events_json: Path | None = None
    no_events_detected: bool = False
    no_haptic_events: bool = False
    gate_categories_used: list[str] = field(default_factory=list)

    def save_all(self) -> dict[str, Path]:
        out: dict[str, Path] = {"source_audio": self.source_wav}
        if self.algorithm_a and self.algorithm_a.exists():
            out["algorithm_a_perception_mapping"] = self.algorithm_a
        if self.algorithm_b and self.algorithm_b.exists():
            out["algorithm_b_frequency_shifting"] = self.algorithm_b
        if self.algorithm_c and self.algorithm_c.exists():
            out["algorithm_c_pitch_matching"] = self.algorithm_c
        if self.algorithm_d and self.algorithm_d.exists():
            out["algorithm_d_haptic_gen"] = self.algorithm_d
        if self.events_json and self.events_json.exists():
            out["events_json"] = self.events_json
        if (
            self.haptic_input_wav is not None
            and self.haptic_input_wav.exists()
            and self.haptic_input_wav != self.source_wav
        ):
            out["gated_audio"] = self.haptic_input_wav
        return out


def _update_events_json_haptics(
    events_json: Path,
    haptic_paths: dict[str, str],
) -> None:
    if not events_json.exists():
        return
    payload = json.loads(events_json.read_text(encoding="utf-8"))
    existing = payload.get("haptic_outputs", {})
    existing.update(haptic_paths)
    payload["haptic_outputs"] = existing
    events_json.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def generate_candidate_tracks(
    input_path: str | Path,
    output_dir: str | Path,
    *,
    from_video: bool = True,
    content_type: str = "game",
    enable_context_detection: bool = True,
    taxonomy_path: str | Path | None = None,
    gate_categories: list[str] | None = None,
) -> CandidateTracks:
    \"\"\"
    Run context detection (optional) then Sound2Hap A–D.

    When gate-eligible events are detected, haptics are stitched onto a full-length
    timeline (one WAV per algorithm). When none match gate_categories, haptic
    generation is skipped.
    \"\"\"
    input_path = Path(input_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    taxonomy = load_taxonomy(taxonomy_path)
    gate_cats = resolve_gate_categories(taxonomy, gate_categories)

    source_wav = output_dir / OUTPUT_NAMES["source_audio"]
    if from_video:
        extract_audio_from_video(input_path, source_wav, sr=INPUT_SR)
    else:
        prepare_source_wav(input_path, source_wav, sr=INPUT_SR)

    events: list[DetectedEvent] | None = None
    events_json_path: Path | None = None
    no_events_detected = False
    no_haptic_events = False
    haptic_input: Path | None = None
    gate_events: list[DetectedEvent] = []

    out_a = output_dir / OUTPUT_NAMES["algorithm_a_perception_mapping"]
    out_b = output_dir / OUTPUT_NAMES["algorithm_b_frequency_shifting"]
    out_c = output_dir / OUTPUT_NAMES["algorithm_c_pitch_matching"]
    out_d = output_dir / OUTPUT_NAMES["algorithm_d_haptic_gen"]

    if enable_context_detection and from_video:
        event_result = detect_events(
            input_path,
            source_wav,
            output_dir,
            taxonomy_path=taxonomy_path,
            write_gated=True,
            gate_categories=gate_categories,
        )
        events = event_result.events
        events_json_path = event_result.events_json
        no_events_detected = event_result.no_events_detected
        no_haptic_events = event_result.no_haptic_events
        gate_events = events_for_haptic_gate(events or [], taxonomy, gate_categories=gate_cats)
        if event_result.gated_wav is not None and event_result.gated_wav.exists():
            haptic_input = event_result.gated_wav
    elif enable_context_detection:
        events_json_path = output_dir / OUTPUT_NAMES["events_json"]
        payload = {
            "no_events_detected": True,
            "no_haptic_events": True,
            "gate_categories_used": gate_cats,
            "timeline_hz": taxonomy.timeline_hz,
            "haptic_outputs": {},
            "events": [],
        }
        events_json_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
        no_events_detected = True
        no_haptic_events = True
    else:
        gate_events = []

    pitch_info = None
    if gate_events:
        stitch_algorithm_output(
            source_wav,
            gate_events,
            out_a,
            percept.process_file,
            process_kwargs={"content": content_type},
        )
        stitch_algorithm_output(source_wav, gate_events, out_b, freq_shift.process_file)
        stitch_algorithm_output(source_wav, gate_events, out_c, pitch_match.process_file)
        stitch_algorithm_output(source_wav, gate_events, out_d, haptic_gen.process_file)

        if events_json_path is not None:
            _update_events_json_haptics(
                events_json_path,
                {
                    "algorithm_a": OUTPUT_NAMES["algorithm_a_perception_mapping"],
                    "algorithm_b": OUTPUT_NAMES["algorithm_b_frequency_shifting"],
                    "algorithm_c": OUTPUT_NAMES["algorithm_c_pitch_matching"],
                    "algorithm_d": OUTPUT_NAMES["algorithm_d_haptic_gen"],
                },
            )

    return CandidateTracks(
        source_wav=source_wav,
        algorithm_a=out_a if gate_events else None,
        algorithm_b=out_b if gate_events else None,
        algorithm_c=out_c if gate_events else None,
        algorithm_d=out_d if gate_events else None,
        output_dir=output_dir,
        haptic_input_wav=haptic_input,
        pitch_match_info=pitch_info,
        events=events,
        events_json=events_json_path,
        no_events_detected=no_events_detected,
        no_haptic_events=no_haptic_events,
        gate_categories_used=gate_cats,
    )
""",
    "utils/__init__.py": """\"\"\"Utils package.\"\"\"
""",
    "utils/normalization.py": """\"\"\"Peak/RMS/loudness normalization (from Sound2Hap).\"\"\"

from __future__ import annotations

import sys
import typing as tp

import torch
import torchaudio


def normalize_loudness(
    wav: torch.Tensor,
    sample_rate: int,
    loudness_headroom_db: float = 14,
    loudness_compressor: bool = False,
    energy_floor: float = 2e-3,
) -> torch.Tensor:
    energy = wav.pow(2).mean().sqrt().item()
    if energy < energy_floor:
        return wav
    transform = torchaudio.transforms.Loudness(sample_rate)
    input_loudness_db = transform(wav).item()
    delta_loudness = -loudness_headroom_db - input_loudness_db
    gain = 10.0 ** (delta_loudness / 20.0)
    output = gain * wav
    if loudness_compressor:
        output = torch.tanh(output)
    return output


def _clip_wav(
    wav: torch.Tensor,
    log_clipping: bool = False,
    stem_name: tp.Optional[str] = None,
) -> None:
    max_scale = wav.abs().max()
    if log_clipping and max_scale > 1:
        clamp_prob = (wav.abs() > 1).float().mean().item()
        print(
            f"CLIPPING {stem_name or ''} happening with proba:",
            clamp_prob,
            "maximum scale:",
            max_scale.item(),
            file=sys.stderr,
        )
    wav.clamp_(-1, 1)


def normalize_audio(
    wav: torch.Tensor,
    normalize: bool = True,
    strategy: str = "peak",
    peak_clip_headroom_db: float = 1,
    peak_normalize_db_clamp: float = 0,
    rms_headroom_db: float = 18,
    loudness_headroom_db: float = 14,
    loudness_compressor: bool = False,
    log_clipping: bool = False,
    sample_rate: tp.Optional[int] = None,
    stem_name: tp.Optional[str] = None,
) -> torch.Tensor:
    scale_peak = 10 ** (-peak_clip_headroom_db / 20)
    normalize_peak = 10 ** (peak_normalize_db_clamp / 20)
    scale_rms = 10 ** (-rms_headroom_db / 20)
    if strategy == "peak":
        wav_max = wav.abs().max()
        rescaling = (scale_peak / wav_max).clamp(max=(normalize_peak / wav_max).clamp(min=1))
        if normalize or rescaling < 1:
            wav = wav * rescaling
    elif strategy == "clip":
        wav = wav.clamp(-scale_peak, scale_peak)
    elif strategy == "rms":
        mono = wav.mean(dim=0)
        rescaling = scale_rms / mono.pow(2).mean().sqrt()
        if normalize or rescaling < 1:
            wav = wav * rescaling
        _clip_wav(wav, log_clipping=log_clipping, stem_name=stem_name)
    elif strategy == "loudness":
        assert sample_rate is not None, "Loudness normalization requires sample rate."
        wav = normalize_loudness(
            wav, sample_rate, loudness_headroom_db, loudness_compressor
        )
        _clip_wav(wav, log_clipping=log_clipping, stem_name=stem_name)
    else:
        assert wav.abs().max() < 1
        assert strategy in ("", "none"), f"Unexpected strategy: '{strategy}'"
    return wav
""",
    "context/taxonomy.yaml": """timeline_hz: 100
context_detector_threshold: 0.85
encoder_threshold: 0.35
impulsive_encoder_threshold: 0.30
# Lower threshold for sustained categories (vehicle, weather) — they score lower than explosions
sustained_encoder_threshold: 0.25
# Minimum seconds between impulsive peaks (separate shots / blasts)
impulsive_min_peak_distance_sec: 0.7
# Half-width used as minimum post-onset content for impulsive gates
impulsive_event_half_width_sec: 0.45
impulsive_onset_search_radius_sec: 0.75
impulsive_decay_tail_sec: 0.85
impulsive_decay_threshold: 0.12
# Short pre-roll before detected onset (gate start)
impulsive_pre_roll_sec: 0.08
# Cap sustained (vehicle, etc.) span in events.json / optional gate
sustained_max_gate_sec: 2.5
# Onset proposal stage (before AST/ViViT)
proposal_rms_hop_ms: 5.0
proposal_threshold_ratio: 0.18
proposal_search_pad_sec: 0.75
proposal_window_sec: 1.0
# Sparse scan for sustained sounds (vehicle, human activity) without sharp onsets
proposal_sustained_hop_sec: 2.0
# Impulsive peak snap: search around classifier hint, snap via spectral flux
impulsive_onset_back_sec: 0.6
impulsive_onset_forward_sec: 1.2
onset_flux_hop_ms: 5.0
onset_flux_min_ratio: 0.45

categories:
  weather:
    audioset_labels:
      - Thunder
      - Rain
      - Wind
      - Rain on surface
      - Storm
    kinetics_labels: []
    context_token_labels:
      - thunder
      - storm
      - rain
    audio_weight: 0.8
    video_weight: 0.2
    # Thunder is an impulsive transient — snap onset via spectral flux
    impulsive: true
    include_in_haptic_gate: true

  gunshot:
    audioset_labels:
      - Gunshot, gunfire
      - Machine gun
      - Cap gun
      - Fusillade
    kinetics_labels:
      - shooting gun
      - playing paintball
    context_token_labels:
      - gunshot
      - gunfire
    audio_weight: 0.7
    video_weight: 0.3
    require_context_or_both: false
    impulsive: true
    include_in_haptic_gate: true

  explosion:
    audioset_labels:
      - Explosion
      - Artillery fire
      - Fireworks
      - Burst, pop
      - Boom
    kinetics_labels:
      - exploding firecrackers
    context_token_labels:
      - explosion
      - boom
      - artillery
      - cannon
    audio_weight: 0.7
    video_weight: 0.3
    require_context_or_both: false
    impulsive: true
    include_in_haptic_gate: true

  vehicle:
    audioset_labels:
      - Vehicle
      - Engine
      - Truck
      - Motor vehicle (road)
      - Idling
      - Tank
    kinetics_labels:
      - driving car
      - riding a bike
      - motorcycling
    context_token_labels:
      - engine_rumble
      - vehicle
      - tank
    audio_weight: 0.75
    video_weight: 0.25
    # Sustained rumble — not impulsive, lower confidence threshold handled by encoder_threshold
    impulsive: false
    include_in_haptic_gate: true

  human_activity:
    audioset_labels:
      - Chainsaw
      - Sawing
      - Wood
    kinetics_labels:
      - chopping wood
      - cutting trees
      - using axe
    context_token_labels:
      - sawing
      - chainsaw
    audio_weight: 0.4
    video_weight: 0.6
    include_in_haptic_gate: false
"""

}

for name, source in FILES.items():
    target = PKG_DIR / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding="utf-8")

sys.path.insert(0, str(PROJECT_ROOT))
print("Installed haptic_gt at", PKG_DIR)
print("Modules:", ", ".join(sorted(FILES)))
from haptic_gt.context.encoders import AST_MODEL_ID, VIVIT_MODEL_ID
print("AST model:", AST_MODEL_ID)
print("ViViT model:", VIVIT_MODEL_ID)


In [ ]:
# Workspace folders on this Colab VM (no Google Drive needed)
from pathlib import Path

WORK_DIR = Path("/content/haptic-workspace")
INPUT_DIR = WORK_DIR / "input"
OUTPUT_DIR = WORK_DIR / "output"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Upload a video in the next cell.")
print("Results will be written to:", OUTPUT_DIR)
print("Download the ZIP at the end — Colab deletes /content when the session ends.")


In [ ]:
from google.colab import files
from pathlib import Path
from IPython.display import Audio, display
from haptic_gt.pipeline import generate_candidate_tracks

print("Choose a video file to upload...")
uploaded = files.upload()
video_name = next(iter(uploaded))
video_path = Path("/content") / video_name

print("Using video:", video_path)
print("Output folder:", OUTPUT_DIR)


In [ ]:
%%time
import json
import soundfile as sf
from pathlib import Path
from haptic_gt.pipeline import OUTPUT_NAMES, generate_candidate_tracks

CONTENT_TYPE = "game"
ENABLE_CONTEXT = True  # set False to skip AST/ViViT and re-run Sound2Hap only
# Categories to include in gated haptics
GATE_CATEGORIES = ["weather", "gunshot", "explosion", "vehicle", "human_activity"]

source_wav = OUTPUT_DIR / OUTPUT_NAMES["source_audio"]
if source_wav.exists() and not ENABLE_CONTEXT:
  print("Reusing existing source audio:", source_wav)
  input_path = source_wav
  from_video = False
else:
  input_path = video_path
  from_video = True

tracks = generate_candidate_tracks(
    input_path,
    OUTPUT_DIR,
    from_video=from_video,
    content_type=CONTENT_TYPE,
    enable_context_detection=ENABLE_CONTEXT,
    gate_categories=GATE_CATEGORIES,
)
saved = tracks.save_all()

src_audio, src_sr = sf.read(saved["source_audio"])
print(f"Source: {len(src_audio)/src_sr:.1f}s @ {src_sr} Hz")
print(f"Haptic input: {tracks.haptic_input_wav}")
print(f"No events detected: {tracks.no_events_detected}")
print(f"No haptic events (gate): {tracks.no_haptic_events}")
print(f"Gate categories: {tracks.gate_categories_used}")
if tracks.events_json and tracks.events_json.exists():
    print(f"Events: {tracks.events_json}")
    print(json.dumps(json.loads(tracks.events_json.read_text()), indent=2)[:2000])

print("Saved files:")
missing = []
for name, path in saved.items():
    ok = path.exists()
    print(f"  {name}: {path} [{'OK' if ok else 'MISSING'}]")
    if not ok:
        missing.append(name)

if tracks.no_haptic_events:
    print("No gate-eligible events — haptic WAVs skipped (see events.json).")
elif missing:
    raise RuntimeError(
        "Generation incomplete — missing outputs: "
        + ", ".join(missing)
        + ". Scroll up for the first error, fix it, then re-run this cell."
    )
print("Generation complete.")


In [ ]:
# Events timeline visualization (run after generation cell)
import json
from pathlib import Path
import matplotlib.pyplot as plt

if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = Path("/content/haptic-workspace/output")

events_path = Path(OUTPUT_DIR) / "events.json"
if events_path.exists():
    data = json.loads(events_path.read_text())
    events = data.get("events", [])
    if events:
        fig, ax = plt.subplots(figsize=(12, 2 + len(events) * 0.3))
        colors = {
            "weather": "tab:blue",
            "gunshot": "tab:red",
            "explosion": "tab:purple",
            "vehicle": "tab:orange",
            "human_activity": "tab:green",
        }
        for i, ev in enumerate(events):
            c = colors.get(ev["category"], "tab:gray")
            ax.barh(i, ev["end_sec"] - ev["start_sec"], left=ev["start_sec"],
                    height=0.6, color=c, alpha=0.7)
            ax.plot(ev["peak_sec"], i, "k|", markersize=12)
            ax.text(ev["end_sec"], i,
                    f" {ev['label']} ({ev['confidence']:.0%})",
                    va="center", fontsize=9)
        ax.set_xlabel("Time (s)")
        ax.set_yticks(range(len(events)))
        ax.set_yticklabels([e["category"] for e in events])
        ax.set_title("Detected events timeline")
        plt.tight_layout()
        plt.show()
    else:
        print("No events in events.json.")
else:
    print("Run the generation cell first to create events.json")


In [ ]:
from pathlib import Path
from IPython.display import Audio, display
from haptic_gt.pipeline import OUTPUT_NAMES

if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = Path("/content/haptic-workspace/output")

labels = {
    "source_audio": "Source audio",
    "algorithm_a_perception_mapping": "A — Perception mapping",
    "algorithm_b_frequency_shifting": "B — Frequency shifting",
    "algorithm_c_pitch_matching": "C — Pitch matching",
    "algorithm_d_haptic_gen": "D — HapticGen",
}

paths = {}
if "saved" in globals():
    paths.update({k: v for k, v in saved.items() if k in labels and v.exists()})
for key in labels:
    if key not in paths:
        candidate = OUTPUT_DIR / OUTPUT_NAMES[key]
        if candidate.exists():
            paths[key] = candidate

missing = [labels[k] for k in labels if k not in paths]
if missing:
    print("Some tracks are missing — re-run the generation cell:")
    for name in missing:
        print(" -", name)
    on_disk = sorted(p.name for p in OUTPUT_DIR.glob("*"))
    print(f"\nCurrently in {OUTPUT_DIR}:", on_disk or "(empty)")
    if not paths:
        raise RuntimeError("No audio outputs found yet. Run the generation cell first.")

for key, label in labels.items():
    if key not in paths:
        continue
    print(label)
    display(Audio(str(paths[key])))


In [ ]:
import shutil
from pathlib import Path
from google.colab import files

if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = Path("/content/haptic-workspace/output")

zip_base = OUTPUT_DIR.parent / "haptic_candidates"
zip_path = Path(shutil.make_archive(str(zip_base), "zip", OUTPUT_DIR))
print("ZIP created:", zip_path)

files.download(str(zip_path))
print("Download started.")


## Human-in-the-loop (next step)

1. Play **source audio** (44.1 kHz) in headphones while feeling each **8 kHz** candidate on haptic hardware.
2. Rate **realism** and **similarity** (e.g. 1–7 Likert) per algorithm — same protocol as [Sound2Hap](https://sound2hap.netlify.app/).
3. If agreement < threshold, tune parameters in `haptic_gt/algorithms/*.py` and re-run.
4. Approved tracks become your **ground truth dataset**.